# NimbusPay Ticket Triage

This notebook combines yesterday's audit and experiment code with today's priority experiment, error analysis, and test prediction workflow. Yesterday's cells are archived below as code and saved output excerpts; they are not executed when running this notebook top to bottom.

The active cells run today's workflow and generate predictions using the selected clean rank-8 adapter.


## Yesterday: audit, baselines, and earlier experiments

### Yesterday's notebook — cell 0

~~~python
from google.colab import files
uploaded = files.upload()  # Select colab_input.zip
~~~

**Saved output excerpt:**

~~~text
<IPython.core.display.HTML object>

Saving colab_input.zip to colab_input.zip

~~~

### Yesterday's notebook — cell 1

~~~python
import zipfile
from pathlib import Path

zip_name = next(name for name in uploaded if name.endswith(".zip"))
ROOT = Path("/content/candidate_pack")

with zipfile.ZipFile(zip_name) as z:
    z.extractall(ROOT)

print("Files:", sorted(str(p.relative_to(ROOT)) for p in ROOT.rglob("*") if p.is_file()))
!nvidia-smi
~~~

**Saved output excerpt:**

~~~text
Files: ['SCHEMA.md', 'data/dev.jsonl', 'data/test_inputs.jsonl', 'data/train.jsonl', 'score.py']
Wed Nov 11 09:47:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+

~~~

### Yesterday's notebook — cell 2

~~~python
import json
import re
from collections import Counter
from datetime import date

train_path = ROOT / "data" / "train.jsonl"
rows = [
    json.loads(line)
    for line in train_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

required_keys = {
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
}

allowed = {
    "category": {"payment_failed", "refund", "fraud", "account_access",
                 "kyc", "offers", "other"},
    "priority": {"P1", "P2", "P3"},
    "channel": {"upi", "card", "netbanking", "wallet", None},
    "language": {"en", "hi-en"},
}

type_rules = {
    "category": (str,),
    "priority": (str,),
    "amount_paise": (int, type(None)),
    "txn_date": (str, type(None)),
    "txn_id": (str, type(None)),
    "channel": (str, type(None)),
    "language": (str,),
    "needs_human": (bool,),  # bool is deliberately checked separately from int
}

bad_json = []
flagged = []
flag_counts = Counter()

for row in rows:
    rid = row["id"]
    ticket = row["messages"][1]["content"]
    raw_label = row["messages"][-1]["content"]

    try:
        label = json.loads(raw_label)
    except Exception as error:
        bad_json.append((rid, str(error), raw_label))
        flag_counts["invalid_json"] += 1
        continue

    flags = []

    if not isinstance(label, dict):
        flags.append("label is not a JSON object")
    else:
        if set(label) != required_keys:
            flags.append(
                f"wrong keys (missing={sorted(required_keys - set(label))}, "
                f"extra={sorted(set(label) - required_keys)})"
            )

        for key, valid_types in type_rules.items():
            if key not in label or type(label[key]) not in valid_types:
                flags.append(f"wrong or missing type/value for {key}")

        for key, values in allowed.items():
            if key in label and label[key] not in values:
                flags.append(f"invalid allowed value for {key}")

        amount = label.get("amount_paise")
        category = label.get("category")
        priority = label.get("priority")

        if type(amount) is int and amount < 0:
            flags.append("negative amount")

        txn_date = label.get("txn_date")
        if txn_date is not None:
            try:
                if date.fromisoformat(txn_date).isoformat() != txn_date:
                    flags.append("date is not YYYY-MM-DD")
            except (TypeError, ValueError):
                flags.append("invalid transaction date")

        txn_id = label.get("txn_id")
        if txn_id is not None and not re.fullmatch(r"NP\d{10}", str(txn_id)):
            flags.append("transaction ID is not NP plus 10 digits")

        # These priority checks flag clear rule conflicts for manual review.
        if category == "fraud" and priority != "P1":
            flags.append("fraud should be P1")

        if type(amount) is int and amount >= 5_000_000 and priority != "P1":
            flags.append("amount is at least ₹50,000 but priority is not P1")

        if (category in {"payment_failed", "refund"}
                and type(amount) is int
                and amount >= 500_000
                and priority == "P3"):
            flags.append("payment failure/refund is at least ₹5,000 but is P3")

        if category == "account_access" and priority == "P3":
            flags.append("account access should be at least P2")

        if priority == "P1" and label.get("needs_human") is False:
            flags.append("P1 but needs_human is false")

    if flags:
        flagged.append((rid, flags, ticket, raw_label))
        for flag in flags:
            # Group related messages so the summary stays readable.
            if flag.startswith("wrong keys"):
                flag_counts["wrong keys"] += 1
            elif "type/value" in flag:
                flag_counts["wrong/missing field type"] += 1
            else:
                flag_counts[flag] += 1

print("Training rows:", len(rows))
print("Invalid JSON rows:", len(bad_json))
print("Rows with one or more audit flags:", len(flagged))
print("\nFlag counts (some rows have multiple flags):")
for name, count in flag_counts.most_common():
    print(f"{name}: {count}")

print("\nInvalid JSON row IDs:")
print([rid for rid, _, _ in bad_json])

print("\nExamples to inspect:")
for rid, flags, ticket, raw_label in flagged[:8]:
    print(f"\nID: {rid}")
    print("Flags:", flags)
    print("Ticket excerpt:", repr(ticket[:350]))
    print("Label:", raw_label)
~~~

**Saved output excerpt:**

~~~text
Training rows: 1445
Invalid JSON rows: 25
Rows with one or more audit flags: 145

Flag counts (some rows have multiple flags):
wrong keys: 70
wrong/missing field type: 70
P1 but needs_human is false: 30
invalid transaction date: 30
invalid_json: 25
transaction ID is not NP plus 10 digits: 20
payment failure/refund is at least ₹5,000 but is P3: 15
amount is at least ₹50,000 but priority is not P1: 9
fraud should be P1: 7
account access should be at least P2: 6

Invalid JSON row IDs:
['tr-00060', 'tr-00095', 'tr-00125', 'tr-00172', 'tr-00267', 'tr-00357', 'tr-00478', 'tr-00497', 'tr-00558', 'tr-00605', 'tr-00674', 'tr-00735', 'tr-00765', 'tr-00809', 'tr-00824', 'tr-00846', 'tr-00958', 'tr-00960', 'tr-00975', 'tr-00978', 'tr-00979', 'tr-00980', 'tr-01070', 'tr-01087', 'tr-01199']

Examples to inspect:

ID: tr-00018
Flags: ['P1 but needs_human is false']
Ticket excerpt: 'Received: 2026-08-01 (Sat)\nSource: email\n\nSubject: Issue with my account\n\nDear NimbusPay support,\nReferral bonus was not received after my friend signed up. Mode: UPI. My phone is a OnePlus Nord. The transaction amount is Rs. 250. Thanks.\n\nRegards,\nImran Khan'
Label: {"category": "offers", "priority": "P1", "amount_paise": 25000, "txn_date": null, "txn_id": null, "channel": "upi", "language": "en", "needs_human": false}

ID: tr-00020
Flags: ["wrong keys (missing=['priority'], extra=['prio'])", 'wrong or missing type/value for priority', 'invalid transaction date']
Ticket excerpt: 'Received: 2026-09-05 (Sat)\nSource: ivr-transcript\n\n[IVR transcript] Hey, How can I check if I have received a refund if it is not yet on my statement? 978 rupees is the amount. I have contacted support about this before. Mode: wallet. Using the Android app. This happened on 02-09-26. Thanks.'
Label: {"category": "refund", "prio": "low", "amount_paise": 97800, "txn_date": "02/09/2026", "txn_id": null, "channel": "wallet", "language": "en", "needs_human": true}

ID: tr-00023
Flags: ['P1 but needs_human is false']
Ticket excerpt: 'Received: 2026-09-15 (Tue)\nSource: app-chat\n\nURGENT!! How can you assist on verifying my identity? I am on the iOS app. Please look into it.'
Label: {"category": "kyc", "priority": "P1", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}

ID: tr-00045
Flags: ['fraud should be P1']
Ticket excerpt: 'Received: 2026-06-01 (Mon)\nSource: email\n\nSubject: Help needed\n\n\nI AM MISSING CASH FROM MY BANK ACCOUNT ACCORDING TO MY APP. The amount was INR 11,594. I am on the iOS app. Waiting for your reply.\n\nRegards,\nFatima Sheikh'
Label: {"category": "fraud", "priority": "P3", "amount_paise": 1159400, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": true}

ID: tr-00049
Flags: ["wrong keys (missing=['priority'], extra=['prio'])", 'wrong or missing type/value for priority']
Ticket excerpt: 'Received: 2026-09-01 (Tue)\nSource: ivr-transcript\n\n[IVR transcript]  Regsitered email id kaise change karun? Dhanyavaad.'
Label: {"category": "other", "prio": "low", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "hi-en", "needs_human": false}

ID: tr-00058
Flags: ["wrong keys (missing=['priority'], extra=['prio'])", 'wrong or missing type/value for priority', 'invalid transaction date']
Ticket excerpt: "Received: 2026-09-13 (Sun)\nSource: ivr-transcript\n\n[IVR transcript] Dear NimbusPay support, I tried to make a transfer and it didn't work. This happened on Aug 26. Thanks."
Label: {"category": "payment_failed", "prio": "low", "amount_paise": null, "txn_date": "26/08/2026", "txn_id": null, "channel": null, "language": "en", "needs_human": false}

ID: tr-00059
Flags: ['P1 but needs_human is false']
Ticket excerpt: 'Received: 2026-08-20 (Thu)\nSource: twitter\n\n@NimbusPayCare A refund for a recent return has not arrived on my card. The amount was 3.7k. App version is 5.5.20. This happened 5 days ago.'
Label: {"category": "refund", "priority": "P1", "amount_paise": 370000, "txn_date": "2026-08-15", "txn_id": null, "channel": "card", "language": "en", "needs_human": false}

ID: tr-00071
Flags: ['transaction ID is not NP plus 10 digits']
Ticket excerpt: 'Received: 2026-08-17 (Mon)\nSource: email\n\nSubject: Problem with NimbusPay\n\nHello ji,\nEk hi order ke do baar paise kat gaye, ek wapas kar do. Reference NP-9718716109 hai. Android app use kar raha hoon. 4 din pehle ki baat hai. Rs. 954 kat gaye. Reply ka wait kar raha hoon.\n\nRegards,\nArjun Das'
Label: {"category": "refund", "priority": "P3", "amount_paise": 95400, "txn_date": "2026-08-13", "txn_id": "NP-9718716109", "channel": null, "language": "hi-en", "needs_human": false}

~~~

### Yesterday's notebook — cell 3

~~~python
import csv

review_path = ROOT / "audit_review.csv"
with review_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "audit_flags", "ticket", "current_label"])
    for rid, flags, ticket, raw_label in flagged:
        writer.writerow([rid, " | ".join(flags), ticket, raw_label])

colab_log = ROOT / "cleaning_log.csv"
with colab_log.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "action", "reason"])

print("Review file:", review_path)
print("Cleaning log template:", colab_log)
print("Flagged rows saved:", len(flagged))
~~~

**Saved output excerpt:**

~~~text
Review file: /content/candidate_pack/audit_review.csv
Cleaning log template: /content/candidate_pack/cleaning_log.csv
Flagged rows saved: 145

~~~

### Yesterday's notebook — cell 4

~~~python
import csv
import json
import re
from datetime import date, datetime
from pathlib import Path

source_path = ROOT / "data" / "train.jsonl"
output_dir = ROOT / "solution"
output_dir.mkdir(exist_ok=True)

cleaned_path = output_dir / "cleaned_train.jsonl"
log_path = output_dir / "cleaning_log.csv"
review_path = output_dir / "manual_review.csv"

required_keys = {
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
}

legal_pattern = re.compile(
    r"\bRBI\b|ombudsman|consumer court|lawyer|legal notice|legal action",
    re.IGNORECASE,
)

repeat_patterns = [
    r"\bthird time\b",
    r"\bsecond complaint\b",
    r"\bthird complaint\b",
    r"\bagain and again\b",
    r"\bcontacted support about this before\b",
    r"\bi have already (?:written|contacted|called|emailed|complained|reported)\b",
    r"\bthis is my \w+ complaint on the same issue\b",
    r"\bmy earlier ticket\b",
    r"\bprevious ticket\b",
    r"\bteesri baar\b",
    r"\bdusri baar\b",
    r"\bbaar\s+baar\b",
    r"\bpehle bhi complaint\b",
    r"\bpichla ticket\b",
]

def is_repeat_contact(ticket):
    return any(
        re.search(pattern, ticket, re.IGNORECASE)
        for pattern in repeat_patterns
    )

def expected_priority(category, amount_paise, ticket):
    if category == "fraud":
        return "P1"

    if isinstance(amount_paise, int) and amount_paise >= 5_000_000:
        return "P1"

    if legal_pattern.search(ticket):
        return "P1"

    if category == "account_access":
        return "P2"

    if (
        category in {"payment_failed", "refund"}
        and isinstance(amount_paise, int)
        and amount_paise >= 500_000
    ):
        return "P2"

    return "P3"

with source_path.open(encoding="utf-8") as f:
    raw_lines = [line for line in f if line.strip()]

kept_rows = []
log_rows = []
manual_rows = []
dropped_count = 0

for raw_line in raw_lines:
    row = json.loads(raw_line)
    ticket = row["messages"][1]["content"]
    raw_label = row["messages"][-1]["content"]
    changes = []

    try:
        label = json.loads(raw_label)
    except json.JSONDecodeError:
        log_rows.append([
            row["id"],
            "drop",
            "Assistant label is truncated or invalid JSON; safe label cannot be recovered."
        ])
        dropped_count += 1
        continue

    if not isinstance(label, dict):
        log_rows.append([
            row["id"],
            "drop",
            "Assistant label is not a JSON object."
        ])
        dropped_count += 1
        continue

    # Replace the corrupted key. Priority is calculated from the schema below.
    if "prio" in label and "priority" not in label:
        del label["prio"]
        label["priority"] = None
        changes.append("replaced corrupted 'prio' with schema-derived 'priority'")

    # Do not guess how to handle any other missing or extra keys.
    missing = required_keys - set(label)
    extra = set(label) - required_keys

    if missing or extra:
        reason = (
            f"Unresolved keys: missing={sorted(missing)}, "
            f"extra={sorted(extra)}; dropped because the intended label is unclear."
        )
        manual_rows.append([row["id"], reason, ticket, raw_label])
        log_rows.append([row["id"], "drop", reason])
        dropped_count += 1
        continue

    category = label["category"]
    amount = label["amount_paise"]

    if not isinstance(category, str) or type(amount) not in (int, type(None)):
        reason = "Category or amount has an invalid type; safe repair is unclear."
        manual_rows.append([row["id"], reason, ticket, raw_label])
        log_rows.append([row["id"], "drop", reason])
        dropped_count += 1
        continue

    # Convert day/month/year dates to the required ISO format.
    old_date = label["txn_date"]

    if isinstance(old_date, str) and "/" in old_date:
        try:
            parsed_date = datetime.strptime(old_date, "%d/%m/%Y").date()

            received_match = re.search(
                r"Received:\s*(\d{4}-\d{2}-\d{2})",
                ticket
            )

            if received_match:
                received_date = date.fromisoformat(received_match.group(1))
                if parsed_date > received_date:
                    raise ValueError("transaction date is after received date")

            label["txn_date"] = parsed_date.isoformat()
            changes.append(
                f"normalized transaction date {old_date} to {label['txn_date']}"
            )

        except ValueError as error:
            reason = f"Could not safely normalize transaction date {old_date}: {error}"
            manual_rows.append([row["id"], reason, ticket, raw_label])
            log_rows.append([row["id"], "drop", reason])
            dropped_count += 1
            continue

    # Remove spaces, hyphens, and other separators from NimbusPay transaction IDs.
    old_id = label["txn_id"]

    if old_id is not None:
        if not isinstance(old_id, str):
            reason = "Transaction ID is not text; safe repair is unclear."
            manual_rows.append([row["id"], reason, ticket, raw_label])
            log_rows.append([row["id"], "drop", reason])
            dropped_count += 1
            continue

        normalized_id = re.sub(r"[^A-Z0-9]", "", old_id.upper())

        if re.fullmatch(r"NP\d{10}", normalized_id):
            if normalized_id != old_id:
                label["txn_id"] = normalized_id
                changes.append(
                    f"normalized transaction ID {old_id} to {normalized_id}"
                )
        else:
            reason = f"Transaction ID could not be safely normalized: {old_id}"
            manual_rows.append([row["id"], reason, ticket, raw_label])
            log_rows.append([row["id"], "drop", reason])
            dropped_count += 1
            continue

    # Recalculate priority using the assignment rules.
    new_priority = expected_priority(category, amount, ticket)

    if label["priority"] != new_priority:
        changes.append(
            f"corrected priority from {label['priority']} to {new_priority}"
        )

    label["priority"] = new_priority

    # P1 always requires a human. Repeat-contact tickets do too.
    # Preserve existing true labels when the phrase detector is uncertain.
    repeat_contact = is_repeat_contact(ticket)

    if new_priority == "P1" or repeat_contact:
        if label["needs_human"] is not True:
            changes.append("set needs_human to true based on P1 or repeat contact")
        label["needs_human"] = True

    elif label["needs_human"] is True:
        manual_rows.append([
            row["id"],
            "Existing needs_human=true was preserved; verify the repeat-contact wording.",
            ticket,
            raw_label,
        ])

    elif label["needs_human"] is not False:
        reason = "needs_human is not a boolean; safe repair is unclear."
        manual_rows.append([row["id"], reason, ticket, raw_label])
        log_rows.append([row["id"], "drop", reason])
        dropped_count += 1
        continue

    row["messages"][-1]["content"] = json.dumps(
        label,
        ensure_ascii=False,
        separators=(",", ":")
    )
    kept_rows.append(row)

    if changes:
        log_rows.append([row["id"], "fix", "; ".join(changes)])

# Write the cleaned copy.
with cleaned_path.open("w", encoding="utf-8") as f:
    for row in kept_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Write the required cleaning log.
with log_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "action", "reason"])
    writer.writerows(log_rows)

# Write a separate file for human review.
with review_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "review_reason", "ticket", "original_label"])
    writer.writerows(manual_rows)

print("Original training rows:", len(raw_lines))
print("Rows kept in cleaned copy:", len(kept_rows))
print("Rows dropped:", dropped_count)
print("Fix/drop log entries:", len(log_rows))
print("Needs manual review:", len(manual_rows))
print("Cleaned training file:", cleaned_path)
print("Cleaning log:", log_path)
print("Manual review file:", review_path)
~~~

**Saved output excerpt:**

~~~text
Original training rows: 1445
Rows kept in cleaned copy: 1420
Rows dropped: 25
Fix/drop log entries: 236
Needs manual review: 4
Cleaned training file: /content/candidate_pack/solution/cleaned_train.jsonl
Cleaning log: /content/candidate_pack/solution/cleaning_log.csv
Manual review file: /content/candidate_pack/solution/manual_review.csv

~~~

### Yesterday's notebook — cell 5

~~~python
import pandas as pd

manual = pd.read_csv(ROOT / "solution" / "manual_review.csv")
print("Rows needing review:", len(manual))
display(manual)
~~~

**Saved output excerpt:**

~~~text
Rows needing review: 4


         id                                      review_reason  \
0  tr-00073  Existing needs_human=true was preserved; verif...   
1  tr-00187  Existing needs_human=true was preserved; verif...   
2  tr-00790  Existing needs_human=true was preserved; verif...   
3  tr-00794  Existing needs_human=true was preserved; verif...   

                                              ticket  \
0  Received: 2026-07-15 (Wed)\nSource: email\n\nS...   
1  Received: 2026-08-04 (Tue)\nSource: email\n\nS...   
2  Received: 2026-08-21 (Fri)\nSource: twitter\n\...   
3  Received: 2026-07-08 (Wed)\nSource: email\n\nt...   

                                      original_label  
0  {"category": "payment_failed", "priority": "P1...  
1  {"category": "payment_failed", "priority": "P1...  
2  {"category": "refund", "priority": "P3", "amou...  
3  {"category": "payment_failed", "priority": "P3...  
~~~

### Yesterday's notebook — cell 6

~~~python
import csv
import json

cleaned_path = ROOT / "solution" / "cleaned_train.jsonl"
log_path = ROOT / "solution" / "cleaning_log.csv"

fixes = {
    "tr-00073": {
        "amount_paise": 16_878_725,
        "priority": "P1",
        "needs_human": True,
        "reason": (
            "Corrected amount: ticket says ₹168,787.25, which is 16,878,725 paise; "
            "priority is P1 because amount is at least ₹50,000."
        ),
    },
    "tr-00187": {
        "amount_paise": 22_000_000,
        "priority": "P1",
        "needs_human": True,
        "reason": (
            "Corrected amount: ticket says 2.2 lakh (₹220,000), which is "
            "22,000,000 paise; priority is P1 because amount is at least ₹50,000."
        ),
    },
}

drops = {
    "tr-00790": "Ticket contains only placeholder text ('asdf asdf'); label cannot be verified.",
    "tr-00794": "Ticket contains only placeholder text ('test test 123'); label cannot be verified.",
}

rows = []
found_ids = set()

with cleaned_path.open(encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        row = json.loads(line)
        rid = row["id"]

        if rid in fixes:
            label = json.loads(row["messages"][-1]["content"])
            for field in ("amount_paise", "priority", "needs_human"):
                label[field] = fixes[rid][field]
            row["messages"][-1]["content"] = json.dumps(
                label, ensure_ascii=False, separators=(",", ":")
            )
            found_ids.add(rid)

        if rid not in drops:
            rows.append(row)
        else:
            found_ids.add(rid)

expected_ids = set(fixes) | set(drops)
if found_ids != expected_ids:
    raise ValueError(f"Expected to find {expected_ids}, but found {found_ids}")

with cleaned_path.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with log_path.open(encoding="utf-8-sig", newline="") as f:
    log_rows = list(csv.DictReader(f))

# Add the manual amount repairs to the existing log entries.
for rid, fix in fixes.items():
    matches = [entry for entry in log_rows if entry["id"] == rid and entry["action"] == "fix"]
    if matches:
        if fix["reason"] not in matches[0]["reason"]:
            matches[0]["reason"] += "; " + fix["reason"]
    else:
        log_rows.append({"id": rid, "action": "fix", "reason": fix["reason"]})

# These two rows are dropped, so remove any earlier fix entry for them.
log_rows = [entry for entry in log_rows if entry["id"] not in drops]
for rid, reason in drops.items():
    log_rows.append({"id": rid, "action": "drop", "reason": reason})

with log_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "action", "reason"])
    writer.writeheader()
    writer.writerows(log_rows)

print("Rows in cleaned training copy:", len(rows))
print("Manually corrected:", sorted(fixes))
print("Dropped as unusable tickets:", sorted(drops))
print("Cleaning log entries:", len(log_rows))
~~~

### Yesterday's notebook — cell 7

~~~python
import json
import re
from collections import Counter
from datetime import date

cleaned_path = ROOT / "solution" / "cleaned_train.jsonl"

required_keys = {
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
}

categories = {
    "payment_failed", "refund", "fraud", "account_access",
    "kyc", "offers", "other"
}
priorities = {"P1", "P2", "P3"}
channels = {"upi", "card", "netbanking", "wallet", None}
languages = {"en", "hi-en"}

legal_pattern = re.compile(
    r"\bRBI\b|ombudsman|consumer court|lawyer|legal notice|legal action",
    re.IGNORECASE,
)

repeat_pattern = re.compile(
    r"\b(second|third|fourth)\s+(time|complaint)\b"
    r"|again and again"
    r"|\bbaar\s+baar\b"
    r"|\bteesri baar\b"
    r"|\bdusri baar\b"
    r"|\bdoosri baar\b"
    r"|pehle bhi complaint"
    r"|pichla ticket"
    r"|previous ticket"
    r"|earlier ticket"
    r"|contacted support about this before"
    r"|already (written|contacted|called|emailed|complained|reported)",
    re.IGNORECASE,
)

def expected_priority(category, amount_paise, ticket):
    if category == "fraud":
        return "P1"
    if type(amount_paise) is int and amount_paise >= 5_000_000:
        return "P1"
    if legal_pattern.search(ticket):
        return "P1"
    if category == "account_access":
        return "P2"
    if (
        category in {"payment_failed", "refund"}
        and type(amount_paise) is int
        and amount_paise >= 500_000
    ):
        return "P2"
    return "P3"

rows = [
    json.loads(line)
    for line in cleaned_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

problems = []
problem_counts = Counter()

for row in rows:
    rid = row["id"]
    ticket = row["messages"][1]["content"]

    try:
        label = json.loads(row["messages"][-1]["content"])
    except Exception:
        problems.append((rid, ["invalid assistant JSON"], ticket))
        problem_counts["invalid assistant JSON"] += 1
        continue

    flags = []

    if not isinstance(label, dict):
        flags.append("assistant label is not an object")
    else:
        if set(label) != required_keys:
            flags.append("wrong or missing keys")

        if label.get("category") not in categories:
            flags.append("invalid category")
        if label.get("priority") not in priorities:
            flags.append("invalid priority")
        if label.get("channel") not in channels:
            flags.append("invalid channel")
        if label.get("language") not in languages:
            flags.append("invalid language")
        if type(label.get("needs_human")) is not bool:
            flags.append("needs_human is not boolean")

        amount = label.get("amount_paise")
        if type(amount) not in (int, type(None)):
            flags.append("amount_paise has invalid type")

        txn_date = label.get("txn_date")
        if txn_date is not None:
            try:
                if date.fromisoformat(txn_date).isoformat() != txn_date:
                    flags.append("transaction date is not YYYY-MM-DD")
            except Exception:
                flags.append("invalid transaction date")

        txn_id = label.get("txn_id")
        if txn_id is not None and (
            not isinstance(txn_id, str)
            or not re.fullmatch(r"NP\d{10}", txn_id)
        ):
            flags.append("invalid transaction ID")

        category = label.get("category")
        if category in {"account_access", "kyc", "other"}:
            txn_fields = ["amount_paise", "txn_date", "txn_id", "channel"]
            if any(label.get(field) is not None for field in txn_fields):
                flags.append("transaction field present for account_access/kyc/other")

        if (
            category in categories
            and label.get("priority") in priorities
            and type(amount) in (int, type(None))
        ):
            wanted = expected_priority(category, amount, ticket)
            if label["priority"] != wanted:
                flags.append(f"priority should be {wanted}")

        repeat_contact = bool(repeat_pattern.search(ticket))
        if label.get("priority") == "P1" and label.get("needs_human") is not True:
            flags.append("P1 but needs_human is not true")
        elif repeat_contact and label.get("needs_human") is not True:
            flags.append("repeat contact but needs_human is not true")
        elif (
            label.get("needs_human") is True
            and label.get("priority") != "P1"
            and not repeat_contact
        ):
            flags.append("needs_human true has no detected reason; review ticket")

    if flags:
        problems.append((rid, flags, ticket))
        for flag in flags:
            problem_counts[flag] += 1

print("Cleaned rows:", len(rows))
print("Rows with one or more issues:", len(problems))
print("Issue counts (a row can have multiple issues):")
for name, count in problem_counts.most_common():
    print(f"{name}: {count}")

print("\nExamples to review:")
for rid, flags, ticket in problems[:8]:
    print(f"\n{rid}: {flags}")
    print(repr(ticket[:300]))
~~~

**Saved output excerpt:**

~~~text
Cleaned rows: 1418
Rows with one or more issues: 0
Issue counts (a row can have multiple issues):

Examples to review:

~~~

### Yesterday's notebook — cell 8

~~~python
%pip install -q transformers accelerate
~~~

### Yesterday's notebook — cell 9

~~~python
import json
import subprocess
import sys
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUT_DIR = ROOT / "solution"
OUT_DIR.mkdir(exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

dev_path = ROOT / "data" / "dev.jsonl"
dev_rows = [
    json.loads(line)
    for line in dev_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
rules_text = (ROOT / "SCHEMA.md").read_text(encoding="utf-8")

def predict_dev(rows, include_rules=False):
    predictions = []

    for row in rows:
        original_messages = row["messages"]
        messages = [
            {"role": original_messages[0]["role"],
             "content": original_messages[0]["content"]},
            {"role": original_messages[1]["role"],
             "content": original_messages[1]["content"]},
        ]

        # This rules-in-prompt version is a reference baseline only.
        if include_rules:
            messages[0]["content"] += (
                "\n\nApply the following triage rules:\n" + rules_text
            )

        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.eos_token_id,
            )

        prompt_length = inputs["input_ids"].shape[1]
        output_text = tokenizer.decode(
            output_ids[0][prompt_length:],
            skip_special_tokens=True,
        )

        predictions.append({
            "id": row["id"],
            "output": output_text,
        })

    return predictions

def save_predictions(predictions, path):
    with path.open("w", encoding="utf-8") as f:
        for item in predictions:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

def score_predictions(pred_path):
    result = subprocess.run(
        [
            sys.executable,
            str(ROOT / "score.py"),
            "--gold", str(dev_path),
            "--pred", str(pred_path),
            "--json",
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    print(result.stdout)

fixed_prompt_path = OUT_DIR / "baseline_fixed_prompt_dev.jsonl"
rules_prompt_path = OUT_DIR / "baseline_rules_prompt_dev.jsonl"

fixed_predictions = predict_dev(dev_rows, include_rules=False)
save_predictions(fixed_predictions, fixed_prompt_path)
print("BASE MODEL WITH FIXED PROMPT")
score_predictions(fixed_prompt_path)

rules_predictions = predict_dev(dev_rows, include_rules=True)
save_predictions(rules_predictions, rules_prompt_path)
print("BASE MODEL WITH RULES IN PROMPT (REFERENCE ONLY)")
score_predictions(rules_prompt_path)
~~~

**Saved output excerpt:**

~~~text
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
WARNING:huggingface_hub.utils._http:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

BASE MODEL WITH FIXED PROMPT
{
 "all": {
  "n": 200,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=other": {
  "n": 16,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=kyc": {
  "n": 24,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=refund": {
  "n": 55,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=payment_failed": {
  "n": 53,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=fraud": {
  "n": 15,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=offers": {
  "n": 18,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=account_access": {
  "n": 19,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 }
}

BASE MODEL WITH RULES IN PROMPT (REFERENCE ONLY)
{
 "all": {
  "n": 200,
  "json_valid": 0.08,
  "exact_match": 0.005,
  "mean_field_acc": 0.0475,
  "fields": {
   "category": 0.07,
   "priority": 0.015,
   "amount_paise": 0.065,
   "txn_date": 0.05,
   "txn_id": 0.065,
   "channel": 0.045,
   "language": 0.05,
   "needs_human": 0.02
  }
 },
 "category=other": {
  "n": 16,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=kyc": {
  "n": 24,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
   "category": 0.0,
   "priority": 0.0,
   "amount_paise": 0.0,
   "txn_date": 0.0,
   "txn_id": 0.0,
   "channel": 0.0,
   "language": 0.0,
   "needs_human": 0.0
  }
 },
 "category=refund": {
  "n": 55,
  "json_valid": 0.05454545454545454,
  "exact_match": 0.01818181818181818,
  "mean_field_acc": 0.038636363636363635,
  "fields": {
   "category": 0.05454545454545454,
   "priority": 0.03636363636363636,
   "amount_paise": 0.03636363636363636,
   "txn_date": 0.03636363636363636,
   "txn_id": 0.03636363636363636,
   "channel": 0.03636363636363636,
   "language": 0.03636363636363636,
   "needs_human": 0.03636363636363636
  }
 },
 "category=payment_failed": {
  "n": 53,
  "json_valid": 0.1509433962264151,
  "exact_match": 0.0,
  "mean_field_acc": 0.07783018867924528,
  "fields": {
   "category": 0.11320754716981132,
   "priority": 0.0,
   "amount_paise": 0.1320754716981132,
   "txn_date": 0.07547169811320754,
   "txn_id": 0.11320754716981132,
   "channel": 0.05660377
~~~

### Yesterday's notebook — cell 10

~~~python
import json

for name in [
    "baseline_fixed_prompt_dev.jsonl",
    "baseline_rules_prompt_dev.jsonl",
]:
    path = ROOT / "solution" / name
    print("\n", name)

    rows = [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    for row in rows[:3]:
        print(row["id"], repr(row["output"][:600]))

    valid = []
    for row in rows:
        try:
            json.loads(row["output"].strip())
            valid.append(row)
        except Exception:
            pass

    if valid:
        print("One valid JSON example:", repr(valid[0]["output"][:600]))
    else:
        print("No valid JSON examples in this file.")
~~~

**Saved output excerpt:**

~~~text

 baseline_fixed_prompt_dev.jsonl
dv-00001 '```json\n{\n  "status": "New",\n  "priority": "High",\n  "category": "Card Activation",\n  "description": "User received an activation request for their card but has not yet been able to use it. They need assistance with activating and using the card.",\n  "ticket_number": "#54047"\n}\n```'
dv-00002 '```json\n{\n  "priority": "normal",\n  "category": "customer_service",\n  "description": "Customer inquires about accepted forms of identification for transactions on their Realme 11 device.",\n  "resolution_required": true,\n  "expected_resolution_timeframe": "within 3 business days"\n}\n```'
dv-00003 '```json\n{\n  "status": "urgent",\n  "priority": "high",\n  "ticket_id": "TICKET_123456",\n  "customer_name": "Customer Name",\n  "contact_number": "+91XXXXXXXXXX",\n  "source": "ivr-transcript",\n  "message": "Dear Customer, we understand your concern regarding the refund of your purchase. We apologize for any inconvenience caused. To proceed with the refund process, please follow these steps:\\n\\n1. Visit the NimbusPay website or app.\\n2. Navigate to the \'My Account\' section.\\n3. Under \'Transactions\', locate the transaction related to your purchase.\\n4. Click on \'Refund Request\'.\\n5. Fill in all nece'
No valid JSON examples in this file.

 baseline_rules_prompt_dev.jsonl
dv-00001 '```json\n{\n  "category": "kyc",\n  "priority": "P3",\n  "amount_paise": null,\n  "txn_date": null,\n  "txn_id": null,\n  "channel": null,\n  "language": "en",\n  "needs_human": true\n}\n```'
dv-00002 '```json\n{\n  "category": "kyc",\n  "priority": "P3",\n  "amount_paise": null,\n  "txn_date": null,\n  "txn_id": null,\n  "channel": null,\n  "language": "en",\n  "needs_human": true\n}\n```'
dv-00003 '```json\n{\n  "category": "refund",\n  "priority": "P1",\n  "amount_paise": 228300,\n  "txn_date": "2026-09-18",\n  "txn_id": "NP1234567890",\n  "channel": "netbanking",\n  "language": "en",\n  "needs_human": true\n}\n```'
One valid JSON example: '{\n  "category": "payment_failed",\n  "priority": "P1",\n  "amount_paise": 496800,\n  "txn_date": "2026-08-01",\n  "txn_id": "NP9512428472",\n  "channel": "netbanking",\n  "language": "en",\n  "needs_human": true\n}'

~~~

### Yesterday's notebook — cell 11

~~~python
import json

cleaned_path = ROOT / "solution" / "cleaned_train.jsonl"

train_rows = [
    json.loads(line)
    for line in cleaned_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

lengths = []

for row in train_rows:
    ids = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=True,
        add_generation_prompt=False,
    )
    lengths.append(len(ids))

lengths_sorted = sorted(lengths)

def percentile(values, p):
    index = int((len(values) - 1) * p)
    return values[index]

print("Training examples:", len(lengths))
print("Median token length:", percentile(lengths_sorted, 0.50))
print("95th percentile:", percentile(lengths_sorted, 0.95))
print("99th percentile:", percentile(lengths_sorted, 0.99))
print("Longest example:", max(lengths))

candidates = [256, 512, 1024, 2048]
sequence_length = next(
    (size for size in candidates if size >= max(lengths)),
    None,
)
print("Suggested sequence length:", sequence_length)
~~~

**Saved output excerpt:**

~~~text
Training examples: 1418
Median token length: 2
95th percentile: 2
99th percentile: 2
Longest example: 2
Suggested sequence length: 256

~~~

### Yesterday's notebook — cell 12

~~~python
import json

cleaned_path = ROOT / "solution" / "cleaned_train.jsonl"

train_rows = [
    json.loads(line)
    for line in cleaned_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

lengths = []

for row in train_rows:
    encoded = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        return_tensors="pt",
    )
    lengths.append(int(encoded["input_ids"].shape[-1]))

lengths_sorted = sorted(lengths)

def percentile(values, p):
    index = int((len(values) - 1) * p)
    return values[index]

print("Training examples:", len(lengths))
print("Median token length:", percentile(lengths_sorted, 0.50))
print("95th percentile:", percentile(lengths_sorted, 0.95))
print("99th percentile:", percentile(lengths_sorted, 0.99))
print("Longest example:", max(lengths))

candidates = [256, 512, 1024, 2048]
sequence_length = next(
    (size for size in candidates if size >= max(lengths)),
    None,
)
print("Suggested sequence length:", sequence_length)
~~~

**Saved output excerpt:**

~~~text
Training examples: 1418
Median token length: 156
95th percentile: 212
99th percentile: 1776
Longest example: 2202
Suggested sequence length: None

~~~

### Yesterday's notebook — cell 13

~~~python
long_examples = sorted(
    zip(lengths, train_rows),
    key=lambda item: item[0],
    reverse=True,
)

print("Examples longer than 2,048 tokens:", sum(n > 2048 for n in lengths))

for token_count, row in long_examples[:10]:
    ticket = row["messages"][1]["content"]
    print(f"\nID: {row['id']} | tokens: {token_count} | ticket characters: {len(ticket)}")
    print(repr(ticket[:500]))
~~~

**Saved output excerpt:**

~~~text
Examples longer than 2,048 tokens: 7

ID: tr-00817 | tokens: 2202 | ticket characters: 8839
'Received: 2026-07-14 (Tue)\nSource: email\n\nSubject: Help needed\n\nHi,\nEk hi order ke do baar paise kat gaye, ek wapas kar do. Android app use kar raha hoon. Card se hua tha. Rakam ₹99,936 hai. Yeh 10-07-26 ko hua. Account mein abhi Rs 3,762.25 pade hain. Teesri baar bol raha hoon. Dhanyavaad.\n\nRegards,\nArjun Das\n\n> On 27 Jun 2026, NimbusPay Support wrote:\n> Thank you for contacting NimbusPay. Your request has been registered with ticket #26806. Our team will get back to you within 48 working hours'

ID: tr-00459 | tokens: 2198 | ticket characters: 8821
'Received: 2026-06-29 (Mon)\nSource: email\n\nSubject: Issue with my account\n\nHello ji,\nShopping ka payment decline ho gaya lekin bank ne paise kaat liye. Txn id NP 2764858720. Total Rs 1,68,444.99 ka tha. App version 6.4.2 hai. Transaction date 27 Jun hai. Please help karo.\n\nRegards,\nRahul Verma\n\n> On 22 Jun 2026, NimbusPay Support wrote:\n> Thank you for contacting NimbusPay. Your request has been registered with ticket #94285. Our team will get back to you within 48 working hours. Please do not sh'

ID: tr-01095 | tokens: 2161 | ticket characters: 8797
'Received: 2026-07-11 (Sat)\nSource: email\n\nSubject: Issue with my account\n\n\nI made a payment and it appears it is showing twice. Transaction date 04/07/2026. It was from the NimbusPay wallet. I am using app version 4.2.4. It is 592 rupees in total.\n\nRegards,\nKavya Reddy\n\n> On 24 Jun 2026, NimbusPay Support wrote:\n> Thank you for contacting NimbusPay. Your request has been registered with ticket #44701. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or car'

ID: tr-00576 | tokens: 2157 | ticket characters: 8805
'Received: 2026-07-06 (Mon)\nSource: email\n\nSubject: Issue with my account\n\nHi team,\nThe referral code I shared did not give my friend the joining bonus. My earlier ticket number is #26726. INR 97 is the amount. Transaction date Jun 24. I am on the iOS app.\n\nRegards,\nKavya Reddy\n\n> On 21 Jun 2026, NimbusPay Support wrote:\n> Thank you for contacting NimbusPay. Your request has been registered with ticket #94206. Our team will get back to you within 48 working hours. Please do not share your OTP, PI'

ID: tr-01233 | tokens: 2118 | ticket characters: 8336
'Received: 2026-07-30 (Thu)\nSource: email\n\nSubject: Complaint\n\nHi team,\nI would like it if I could get an item refunded.  I am using app version 7.9.13.  It was a card transaction.  Amount: Rs 11,755.75 Kindly resolve this asap.\n\nRegards,\nRohan Gupta\n\n> On 20 Jul 2026, NimbusPay Support wrote:\n> Thank you for contacting NimbusPay.  Your request has been registered with ticket #11314.  Our team will get back to you within 48 working hours.  Please do not share your OTP, PIN or card details with an'

ID: tr-00524 | tokens: 2106 | ticket characters: 8702
'Received: 2026-08-04 (Tue)\nSource: email\n\nSubject: Problem with NimbusPay\n\nGood morning,\nWhat is necessary for the identity verification? Please help.\n\nRegards,\nFatima Sheikh\n\n> On 16 Jul 2026, NimbusPay Support wrote:\n> Thank you for contacting NimbusPay. Your request has been registered with ticket #19234. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or card details with anyone, including people who claim to be from NimbusPay. You can track the statu'

ID: tr-00654 | tokens: 2056 | ticket characters: 8269
'Received: 2026-08-20 (Thu)\nSource: email\n\nSubject: Help needed\n\n\nWhere do I go for a refund? Rs. 1827 is the amount. I will go to consumer court. Using the Android app. Mode: internet banking. This happened on 19th August 2026. Ref: np6969356191 Thanks.\n\nRegards,\nImran Khan\n\n> On 09 Aug 2026, NimbusPay Support wrote:\n> Thank you for contacting NimbusPay. Your request has been registered with ticket #21263. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN o'

ID: tr-00566 | tokens: 2048 | ticket characters: 8225
'Received: 2026-08-19 (Wed)\nSource: email\n\nSubject: Help needed\n\nHi,\nI need to cancel a purchase? Dated 13th August 2026. Transaction ID NP1384563217. It was a netbanking transaction. It is INR 5,924 in total.\n\nRegards,\nRohan Gupta\n\n> On 14 Aug 2026, NimbusPay Support wrote:\n> Thank you for contacting NimbusPay. Your request has been registered with ticket #47579. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or card details with anyone, including people'

ID: tr-01333 | tokens: 2047 | ticket characters: 8252
'Received: 2026-08-05 (Wed)\nSource: email\n\nSubject: Problem with NimbusPay\n\nHello,\nI have a charge I did not make. Transaction ID np4076009763. It took place today. Amount: ₹48,821 It was a netbanking transaction. Please look into it.\n\nRegards,\nFatima Sheikh\n\n> On 03 Aug 2026, NimbusPay Support wrote:\n> Thank you for contacting Nimbus
~~~

### Yesterday's notebook — cell 14

~~~python
%pip install -q bitsandbytes peft
~~~

### Yesterday's notebook — cell 15

~~~python
import json
import torch

MAX_LENGTH = 2304
cleaned_path = ROOT / "solution" / "cleaned_train.jsonl"

cleaned_rows = [
    json.loads(line)
    for line in cleaned_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

def get_token_ids(messages):
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        return_tensors="pt",
    )
    return encoded["input_ids"][0].tolist()

train_examples = []
lengths = []

for row in cleaned_rows:
    messages = row["messages"]
    prompt_messages = messages[:-1]

    prompt_ids = get_token_ids(prompt_messages)
    full_ids = get_token_ids(messages)

    # Confirm the ticket prompt is exactly the beginning of the full example.
    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError(f"Prompt/answer boundary mismatch in {row['id']}")

    if len(full_ids) > MAX_LENGTH:
        raise ValueError(
            f"{row['id']} has {len(full_ids)} tokens, longer than MAX_LENGTH={MAX_LENGTH}"
        )

    if len(full_ids) <= len(prompt_ids):
        raise ValueError(f"No assistant answer tokens found in {row['id']}")

    # -100 means: don't calculate training loss on these prompt tokens.
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    train_examples.append({
        "input_ids": full_ids,
        "labels": labels,
    })
    lengths.append(len(full_ids))

pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id

def collate_batch(batch):
    batch_max = max(len(item["input_ids"]) for item in batch)

    input_ids = torch.full(
        (len(batch), batch_max),
        pad_id,
        dtype=torch.long,
    )
    attention_mask = torch.zeros(
        (len(batch), batch_max),
        dtype=torch.long,
    )
    labels = torch.full(
        (len(batch), batch_max),
        -100,
        dtype=torch.long,
    )

    for i, item in enumerate(batch):
        size = len(item["input_ids"])
        input_ids[i, :size] = torch.tensor(item["input_ids"])
        attention_mask[i, :size] = 1
        labels[i, :size] = torch.tensor(item["labels"])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

example = train_examples[0]
num_masked = sum(token == -100 for token in example["labels"])
num_answer_tokens = len(example["labels"]) - num_masked

print("Prepared examples:", len(train_examples))
print("Longest prepared example:", max(lengths))
print("Sequence limit:", MAX_LENGTH)
print("Example prompt tokens masked from loss:", num_masked)
print("Example assistant tokens used for loss:", num_answer_tokens)
~~~

**Saved output excerpt:**

~~~text
Prepared examples: 1418
Longest prepared example: 2202
Sequence limit: 2304
Example prompt tokens masked from loss: 130
Example assistant tokens used for loss: 71

~~~

### Yesterday's notebook — cell 16

~~~python
import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Release the original 16-bit baseline model to make GPU memory available.
if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=torch.float16,
    device_map="auto",
)

base_model.config.use_cache = False

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()

trainable = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
print("Trainable parameters:", trainable)
print("Under 50M limit:", trainable <= 50_000_000)
print("GPU:", torch.cuda.get_device_name(0))
~~~

**Saved output excerpt:**

~~~text
Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Trainable parameters: 18464768
Under 50M limit: True
GPU: Tesla T4

~~~

### Yesterday's notebook — cell 17

~~~python
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/nimbuspay_solution")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Keep the cleaned data, log, and baseline predictions in your Drive.
for name in [
    "cleaned_train.jsonl",
    "cleaning_log.csv",
    "baseline_fixed_prompt_dev.jsonl",
    "baseline_rules_prompt_dev.jsonl",
]:
    source = ROOT / "solution" / name
    if source.exists():
        shutil.copy2(source, DRIVE_ROOT / name)

print("Saved project files to:", DRIVE_ROOT)
~~~

**Saved output excerpt:**

~~~text
Mounted at /content/drive
Saved project files to: /content/drive/MyDrive/nimbuspay_solution

~~~

### Yesterday's notebook — cell 18

~~~python
import json
import time
import torch
from transformers import Trainer, TrainingArguments

RUN_DIR = DRIVE_ROOT / "runs" / "clean_qlora_r16_e2"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ADAPTER_DIR = RUN_DIR / "adapter"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

# Batch size 1 is safer for a 2,304-token limit on a T4.
# Accumulating 16 batches gives an effective batch size of 16.
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_examples,
    data_collator=collate_batch,
)

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

train_result = trainer.train()

training_minutes = (time.time() - start_time) / 60
peak_allocated_gb = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
peak_reserved_gb = torch.cuda.max_memory_reserved(0) / (1024 ** 3)

trainer.model.save_pretrained(
    ADAPTER_DIR,
    safe_serialization=True,
)

run_summary = {
    "model_id": MODEL_ID,
    "data": "cleaned_train.jsonl",
    "method": "4-bit QLoRA",
    "rank": 16,
    "epochs": 2,
    "sequence_length": MAX_LENGTH,
    "learning_rate": 2e-4,
    "trainable_parameters": trainable,
    "training_minutes": training_minutes,
    "peak_gpu_allocated_gb": peak_allocated_gb,
    "peak_gpu_reserved_gb": peak_reserved_gb,
    "trainer_metrics": train_result.metrics,
    "training_log": trainer.state.log_history,
}

with (RUN_DIR / "run_summary.json").open("w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, default=str)

print("Training minutes:", round(training_minutes, 1))
print("Peak GPU memory allocated (GB):", round(peak_allocated_gb, 2))
print("Peak GPU memory reserved (GB):", round(peak_reserved_gb, 2))
print("Adapter saved to:", ADAPTER_DIR)
print("Summary saved to:", RUN_DIR / "run_summary.json")
~~~

**Saved output excerpt:**

~~~text
<IPython.core.display.HTML object>

Training minutes: 34.7
Peak GPU memory allocated (GB): 6.22
Peak GPU memory reserved (GB): 14.22
Adapter saved to: /content/drive/MyDrive/nimbuspay_solution/runs/clean_qlora_r16_e2/adapter
Summary saved to: /content/drive/MyDrive/nimbuspay_solution/runs/clean_qlora_r16_e2/run_summary.json

~~~

### Yesterday's notebook — cell 19

~~~python
import json
import subprocess
import sys
import torch
from tqdm.auto import tqdm

model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

dev_rows = [
    json.loads(line)
    for line in (ROOT / "data" / "dev.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

dev_predictions = []

for row in tqdm(dev_rows, desc="Scoring fine-tuned model on dev"):
    messages = row["messages"][:2]  # fixed system prompt and customer ticket

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[1]
    output_text = tokenizer.decode(
        output_ids[0][prompt_length:],
        skip_special_tokens=True,
    )

    dev_predictions.append({
        "id": row["id"],
        "output": output_text,
    })

dev_pred_path = DRIVE_ROOT / "clean_qlora_r16_e2_dev.jsonl"

with dev_pred_path.open("w", encoding="utf-8") as f:
    for item in dev_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "score.py"),
        "--gold", str(ROOT / "data" / "dev.jsonl"),
        "--pred", str(dev_pred_path),
        "--json",
    ],
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)
print("Dev predictions saved to:", dev_pred_path)
~~~

**Saved output excerpt:**

~~~text
Scoring fine-tuned model on dev:   0%|          | 0/200 [00:00<?, ?it/s]

{
 "all": {
  "n": 200,
  "json_valid": 1.0,
  "exact_match": 0.775,
  "mean_field_acc": 0.9625,
  "fields": {
   "category": 0.97,
   "priority": 0.89,
   "amount_paise": 0.94,
   "txn_date": 0.98,
   "txn_id": 1.0,
   "channel": 0.99,
   "language": 1.0,
   "needs_human": 0.93
  }
 },
 "category=other": {
  "n": 16,
  "json_valid": 1.0,
  "exact_match": 0.625,
  "mean_field_acc": 0.9296875,
  "fields": {
   "category": 0.8125,
   "priority": 0.875,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 0.9375,
   "language": 1.0,
   "needs_human": 0.8125
  }
 },
 "category=kyc": {
  "n": 24,
  "json_valid": 1.0,
  "exact_match": 1.0,
  "mean_field_acc": 1.0,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=refund": {
  "n": 55,
  "json_valid": 1.0,
  "exact_match": 0.7454545454545455,
  "mean_field_acc": 0.9590909090909091,
  "fields": {
   "category": 1.0,
   "priority": 0.8363636363636363,
   "amount_paise": 0.9454545454545454,
   "txn_date": 0.9818181818181818,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.9090909090909091
  }
 },
 "category=payment_failed": {
  "n": 53,
  "json_valid": 1.0,
  "exact_match": 0.7358490566037735,
  "mean_field_acc": 0.9599056603773585,
  "fields": {
   "category": 0.9811320754716981,
   "priority": 0.8490566037735849,
   "amount_paise": 0.9245283018867925,
   "txn_date": 0.9811320754716981,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.9433962264150944
  }
 },
 "category=fraud": {
  "n": 15,
  "json_valid": 1.0,
  "exact_match": 0.6,
  "mean_field_acc": 0.9166666666666666,
  "fields": {
   "category": 0.8666666666666667,
   "priority": 0.8666666666666667,
   "amount_paise": 0.7333333333333333,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 0.9333333333333333,
   "language": 1.0,
   "needs_human": 0.9333333333333333
  }
 },
 "category=offers": {
  "n": 18,
  "json_valid": 1.0,
  "exact_match": 0.8333333333333334,
  "mean_field_acc": 0.9722222222222222,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 0.9444444444444444,
   "txn_date": 0.8888888888888888,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.9444444444444444
  }
 },
 "category=account_access": {
  "n": 19,
  "json_valid": 1.0,
  "exact_match": 0.8947368421052632,
  "mean_field_acc": 0.9868421052631579,
  "fields": {
   "category": 1.0,
   "priority": 0.9473684210526315,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.9473684210526315
  }
 }
}

Dev predictions saved to: /content/drive/MyDrive/nimbuspay_solution/clean_qlora_r16_e2_dev.jsonl

~~~

### Yesterday's notebook — cell 20

~~~python
import json
from collections import Counter

pred_path = DRIVE_ROOT / "clean_qlora_r16_e2_dev.jsonl"
gold_path = ROOT / "data" / "dev.jsonl"

gold_rows = [
    json.loads(line)
    for line in gold_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
pred_rows = {
    item["id"]: item["output"]
    for item in (
        json.loads(line)
        for line in pred_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    )
}

fields = [
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
]

error_counts = Counter()
failures = []

for row in gold_rows:
    target = json.loads(row["messages"][-1]["content"])
    prediction = json.loads(pred_rows[row["id"]].strip())
    wrong = [field for field in fields if prediction.get(field) != target[field]]

    if wrong:
        error_counts.update(wrong)
        failures.append((row, target, prediction, wrong))

print("Dev tickets with at least one wrong field:", len(failures))
print("Wrong fields by count:", dict(error_counts))

for row, target, prediction, wrong in failures[:10]:
    print(f"\nID: {row['id']} | wrong fields: {wrong}")
    print("Ticket:", repr(row["messages"][1]["content"][:450]))
    print("Expected:", {field: target[field] for field in wrong})
    print("Predicted:", {field: prediction[field] for field in wrong})
~~~

**Saved output excerpt:**

~~~text
Dev tickets with at least one wrong field: 45
Wrong fields by count: {'needs_human': 14, 'amount_paise': 12, 'priority': 22, 'category': 6, 'channel': 2, 'txn_date': 4}

ID: dv-00001 | wrong fields: ['needs_human']
Ticket: 'Received: 2026-08-28 (Fri)\nSource: app-chat\n\nGood morning, What needs to be done to activate and use a card? My earlier ticket number is #54047. Please look into it.'
Expected: {'needs_human': False}
Predicted: {'needs_human': True}

ID: dv-00007 | wrong fields: ['amount_paise']
Ticket: "Received: 2026-08-16 (Sun)\nSource: app-chat\n\nI have a payment that I didn't make. ₹877 is the amount. The txn id is NP-0755176283. It was a UPI transaction. Please help."
Expected: {'amount_paise': 87700}
Predicted: {'amount_paise': 877}

ID: dv-00011 | wrong fields: ['priority', 'needs_human']
Ticket: 'Received: 2026-06-17 (Wed)\nSource: email\n\nSubject: Problem with NimbusPay\n\nURGENT!!\nWhy am I missing my refund. App version is 4.2.7. ₹84,137 was the value. This happened yesterday. ID: np3333133441\n\nRegards,\nPooja Nair'
Expected: {'priority': 'P1', 'needs_human': True}
Predicted: {'priority': 'P2', 'needs_human': False}

ID: dv-00021 | wrong fields: ['priority']
Ticket: 'Received: 2026-06-26 (Fri)\nSource: app-chat\n\nGood morning, I was supposed to get a refund but I do not see it. Rs. 15775 is the amount. Mode: wallet. Transaction reference NP 4112795249. Dated 15/06/2026. Please look into it.'
Expected: {'priority': 'P2'}
Predicted: {'priority': 'P3'}

ID: dv-00022 | wrong fields: ['amount_paise']
Ticket: "Received: 2026-08-05 (Wed)\nSource: email\n\nSubject: Problem with NimbusPay\n\nHey,\nThere are a couple payments with my card shown that I'm not responsible for. What is going on? Maybe someone has access to my account. Dated Aug 5. The amount was Rs. 1677. I am on the iOS app. Thanks.\n\nRegards,\nAnanya Bose"
Expected: {'amount_paise': 167700}
Predicted: {'amount_paise': 1677}

ID: dv-00030 | wrong fields: ['needs_human']
Ticket: 'Received: 2026-08-22 (Sat)\nSource: email\n\nSubject: Issue with my account\n\nHello,\nWhat currencies can I hold? My earlier ticket number is #45653.\n\nRegards,\nFatima Sheikh'
Expected: {'needs_human': False}
Predicted: {'needs_human': True}

ID: dv-00034 | wrong fields: ['priority', 'needs_human']
Ticket: 'Received: 2026-08-22 (Sat)\nSource: twitter\n\n@NimbusPayCare Shopping ka payment decline ho gaya lekin bank ne paise kaat liye. Rs. 36737 ka transaction tha. Txn id np 94424 25625. Wallet se hua tha.'
Expected: {'priority': 'P2', 'needs_human': False}
Predicted: {'priority': 'P1', 'needs_human': True}

ID: dv-00035 | wrong fields: ['priority']
Ticket: 'Received: 2026-07-09 (Thu)\nSource: email\n\nSubject: Help needed\n\nHi,\nI got charged twice for the same payment. Please removed the pending payment. Thank you. Transaction date 22-06-26. Mode: wallet. Transaction ID np 82704 11138. ₹6,686 is the amount. Please look into it.\n\nRegards,\nRahul Verma'
Expected: {'priority': 'P2'}
Predicted: {'priority': 'P3'}

ID: dv-00037 | wrong fields: ['priority', 'needs_human']
Ticket: 'Received: 2026-09-07 (Mon)\nSource: app-chat\n\nMy transaction history is looking a little off. I have been charged twice for multiple different purchases, and would like to have those duplicate charges refunded. The txn id is np 42846 07517. Transaction date 6 Sep. 284669 rupees was the value. Waiting for your reply.'
Expected: {'priority': 'P1', 'needs_human': True}
Predicted: {'priority': 'P3', 'needs_human': False}

ID: dv-00042 | wrong fields: ['category', 'channel']
Ticket: "Received: 2026-08-28 (Fri)\nSource: email\n\nSubject: Complaint\n\nDear NimbusPay support,\nOn the card that is coming, what's the tracking info? This is my second complaint on the same issue. Using the Android app.\n\nRegards,\nImran Khan"
Expected: {'category': 'other', 'channel': None}
Predicted: {'category': 'refund', 'channel': 'card'}

~~~

### Yesterday's notebook — cell 21

~~~python
import json

raw_path = ROOT / "data" / "train.jsonl"

raw_rows = [
    json.loads(line)
    for line in raw_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

raw_train_examples = []
raw_lengths = []
invalid_raw_labels = 0

for row in raw_rows:
    messages = row["messages"]
    prompt_messages = messages[:-1]

    prompt_encoded = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        return_tensors="pt",
    )
    full_encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        return_tensors="pt",
    )

    prompt_ids = prompt_encoded["input_ids"][0].tolist()
    full_ids = full_encoded["input_ids"][0].tolist()

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError(f"Prompt/answer boundary mismatch in {row['id']}")

    if len(full_ids) > MAX_LENGTH:
        raise ValueError(
            f"{row['id']} has {len(full_ids)} tokens, exceeding {MAX_LENGTH}"
        )

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    raw_train_examples.append({
        "input_ids": full_ids,
        "labels": labels,
    })
    raw_lengths.append(len(full_ids))

    try:
        json.loads(messages[-1]["content"])
    except Exception:
        invalid_raw_labels += 1

print("Raw training examples:", len(raw_train_examples))
print("Invalid assistant labels in raw data:", invalid_raw_labels)
print("Longest raw example:", max(raw_lengths))
print("Sequence limit:", MAX_LENGTH)
~~~

**Saved output excerpt:**

~~~text
Raw training examples: 1445
Invalid assistant labels in raw data: 25
Longest raw example: 2215
Sequence limit: 2304

~~~

### Yesterday's notebook — cell 22

~~~python
import gc
import json
import time
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# Release the cleaned-data model so a fresh base model can be loaded.
for name in ["trainer", "model", "base_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

quant_config_raw = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model_raw = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config_raw,
    dtype=torch.float16,
    device_map="auto",
)
base_model_raw.config.use_cache = False
base_model_raw = prepare_model_for_kbit_training(
    base_model_raw,
    use_gradient_checkpointing=True,
)

lora_config_raw = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model_raw = get_peft_model(base_model_raw, lora_config_raw)
model_raw.print_trainable_parameters()

raw_trainable = sum(
    p.numel() for p in model_raw.parameters() if p.requires_grad
)
print("Raw-run trainable parameters:", raw_trainable)

RAW_RUN_DIR = DRIVE_ROOT / "runs" / "raw_qlora_r16_e2"
RAW_CHECKPOINT_DIR = RAW_RUN_DIR / "checkpoints"
RAW_ADAPTER_DIR = RAW_RUN_DIR / "adapter"
RAW_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RAW_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

raw_args = TrainingArguments(
    output_dir=str(RAW_CHECKPOINT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42,
)

trainer_raw = Trainer(
    model=model_raw,
    args=raw_args,
    train_dataset=raw_train_examples,
    data_collator=collate_batch,
)

torch.cuda.reset_peak_memory_stats()
raw_start = time.time()

raw_train_result = trainer_raw.train()

raw_minutes = (time.time() - raw_start) / 60
raw_peak_allocated = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
raw_peak_reserved = torch.cuda.max_memory_reserved(0) / (1024 ** 3)

trainer_raw.model.save_pretrained(
    RAW_ADAPTER_DIR,
    safe_serialization=True,
)

raw_summary = {
    "model_id": MODEL_ID,
    "data": "raw train.jsonl",
    "method": "4-bit QLoRA",
    "rank": 16,
    "epochs": 2,
    "sequence_length": MAX_LENGTH,
    "learning_rate": 2e-4,
    "trainable_parameters": raw_trainable,
    "training_minutes": raw_minutes,
    "peak_gpu_allocated_gb": raw_peak_allocated,
    "peak_gpu_reserved_gb": raw_peak_reserved,
    "trainer_metrics": raw_train_result.metrics,
    "training_log": trainer_raw.state.log_history,
}

with (RAW_RUN_DIR / "run_summary.json").open("w", encoding="utf-8") as f:
    json.dump(raw_summary, f, indent=2, default=str)

print("Raw-run training minutes:", round(raw_minutes, 1))
print("Peak GPU memory allocated (GB):", round(raw_peak_allocated, 2))
print("Peak GPU memory reserved (GB):", round(raw_peak_reserved, 2))
print("Raw adapter saved to:", RAW_ADAPTER_DIR)
print("Summary saved to:", RAW_RUN_DIR / "run_summary.json")
~~~

**Saved output excerpt:**

~~~text
Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Raw-run trainable parameters: 18464768


<IPython.core.display.HTML object>

Raw-run training minutes: 35.0
Peak GPU memory allocated (GB): 7.11
Peak GPU memory reserved (GB): 14.2
Raw adapter saved to: /content/drive/MyDrive/nimbuspay_solution/runs/raw_qlora_r16_e2/adapter
Summary saved to: /content/drive/MyDrive/nimbuspay_solution/runs/raw_qlora_r16_e2/run_summary.json

~~~

### Yesterday's notebook — cell 23

~~~python
import json
import subprocess
import sys
import torch
from tqdm.auto import tqdm

model_raw.eval()
model_raw.gradient_checkpointing_disable()
model_raw.config.use_cache = True

raw_dev_predictions = []

for row in tqdm(dev_rows, desc="Scoring raw-data adapter on dev"):
    messages = row["messages"][:2]  # fixed prompt and customer ticket

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model_raw.device)

    with torch.inference_mode():
        output_ids = model_raw.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[1]
    output_text = tokenizer.decode(
        output_ids[0][prompt_length:],
        skip_special_tokens=True,
    )

    raw_dev_predictions.append({
        "id": row["id"],
        "output": output_text,
    })

raw_dev_path = DRIVE_ROOT / "raw_qlora_r16_e2_dev.jsonl"

with raw_dev_path.open("w", encoding="utf-8") as f:
    for item in raw_dev_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

score_result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "score.py"),
        "--gold", str(ROOT / "data" / "dev.jsonl"),
        "--pred", str(raw_dev_path),
        "--json",
    ],
    capture_output=True,
    text=True,
    check=True,
)

print(score_result.stdout)
print("Raw-run dev predictions saved to:", raw_dev_path)
~~~

**Saved output excerpt:**

~~~text
Scoring raw-data adapter on dev:   0%|          | 0/200 [00:00<?, ?it/s]

{
 "all": {
  "n": 200,
  "json_valid": 1.0,
  "exact_match": 0.62,
  "mean_field_acc": 0.938125,
  "fields": {
   "category": 0.985,
   "priority": 0.655,
   "amount_paise": 0.99,
   "txn_date": 0.985,
   "txn_id": 1.0,
   "channel": 0.995,
   "language": 1.0,
   "needs_human": 0.895
  }
 },
 "category=other": {
  "n": 16,
  "json_valid": 1.0,
  "exact_match": 0.8125,
  "mean_field_acc": 0.96875,
  "fields": {
   "category": 0.9375,
   "priority": 0.8125,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=kyc": {
  "n": 24,
  "json_valid": 1.0,
  "exact_match": 1.0,
  "mean_field_acc": 1.0,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=refund": {
  "n": 55,
  "json_valid": 1.0,
  "exact_match": 0.36363636363636365,
  "mean_field_acc": 0.8977272727272727,
  "fields": {
   "category": 1.0,
   "priority": 0.41818181818181815,
   "amount_paise": 0.9636363636363636,
   "txn_date": 0.9818181818181818,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.8181818181818182
  }
 },
 "category=payment_failed": {
  "n": 53,
  "json_valid": 1.0,
  "exact_match": 0.4528301886792453,
  "mean_field_acc": 0.9056603773584906,
  "fields": {
   "category": 0.9811320754716981,
   "priority": 0.49056603773584906,
   "amount_paise": 1.0,
   "txn_date": 0.9811320754716981,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.7924528301886793
  }
 },
 "category=fraud": {
  "n": 15,
  "json_valid": 1.0,
  "exact_match": 0.8,
  "mean_field_acc": 0.9666666666666667,
  "fields": {
   "category": 0.9333333333333333,
   "priority": 0.9333333333333333,
   "amount_paise": 1.0,
   "txn_date": 0.9333333333333333,
   "txn_id": 1.0,
   "channel": 0.9333333333333333,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=offers": {
  "n": 18,
  "json_valid": 1.0,
  "exact_match": 0.9444444444444444,
  "mean_field_acc": 0.9930555555555556,
  "fields": {
   "category": 1.0,
   "priority": 0.9444444444444444,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=account_access": {
  "n": 19,
  "json_valid": 1.0,
  "exact_match": 0.7368421052631579,
  "mean_field_acc": 0.9671052631578947,
  "fields": {
   "category": 1.0,
   "priority": 0.7368421052631579,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 }
}

Raw-run dev predictions saved to: /content/drive/MyDrive/nimbuspay_solution/raw_qlora_r16_e2_dev.jsonl

~~~

### Yesterday's notebook — cell 24

~~~python
import gc
import json
import time
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# Release the raw-data QLoRA model and its trainer.
for name in ["trainer_raw", "model_raw", "base_model_raw"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Load the same base model in 16-bit, without 4-bit quantization.
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)
base_model_fp16.config.use_cache = False
base_model_fp16.gradient_checkpointing_enable()
base_model_fp16.enable_input_require_grads()

lora_config_fp16 = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model_fp16 = get_peft_model(base_model_fp16, lora_config_fp16)
model_fp16.print_trainable_parameters()

fp16_trainable = sum(
    p.numel() for p in model_fp16.parameters() if p.requires_grad
)
print("Trainable parameters:", fp16_trainable)
print("Under 50M limit:", fp16_trainable <= 50_000_000)

FP16_RUN_DIR = DRIVE_ROOT / "runs" / "clean_lora_fp16_r16_e2"
FP16_CHECKPOINT_DIR = FP16_RUN_DIR / "checkpoints"
FP16_ADAPTER_DIR = FP16_RUN_DIR / "adapter"
FP16_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FP16_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

fp16_args = TrainingArguments(
    output_dir=str(FP16_CHECKPOINT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42,
)

trainer_fp16 = Trainer(
    model=model_fp16,
    args=fp16_args,
    train_dataset=train_examples,  # same cleaned data as the QLoRA reference
    data_collator=collate_batch,
)

torch.cuda.reset_peak_memory_stats()
fp16_start = time.time()

fp16_train_result = trainer_fp16.train()

fp16_minutes = (time.time() - fp16_start) / 60
fp16_peak_allocated = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
fp16_peak_reserved = torch.cuda.max_memory_reserved(0) / (1024 ** 3)

trainer_fp16.model.save_pretrained(
    FP16_ADAPTER_DIR,
    safe_serialization=True,
)

fp16_summary = {
    "model_id": MODEL_ID,
    "data": "cleaned_train.jsonl",
    "method": "16-bit LoRA",
    "rank": 16,
    "epochs": 2,
    "sequence_length": MAX_LENGTH,
    "learning_rate": 2e-4,
    "trainable_parameters": fp16_trainable,
    "training_minutes": fp16_minutes,
    "peak_gpu_allocated_gb": fp16_peak_allocated,
    "peak_gpu_reserved_gb": fp16_peak_reserved,
    "trainer_metrics": fp16_train_result.metrics,
    "training_log": trainer_fp16.state.log_history,
}

with (FP16_RUN_DIR / "run_summary.json").open("w", encoding="utf-8") as f:
    json.dump(fp16_summary, f, indent=2, default=str)

print("16-bit LoRA training minutes:", round(fp16_minutes, 1))
print("Peak GPU memory allocated (GB):", round(fp16_peak_allocated, 2))
print("Peak GPU memory reserved (GB):", round(fp16_peak_reserved, 2))
print("Adapter saved to:", FP16_ADAPTER_DIR)
print("Summary saved to:", FP16_RUN_DIR / "run_summary.json")
~~~

**Saved output excerpt:**

~~~text
Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]
~~~

### Yesterday's notebook — cell 25

~~~python
from peft.import_utils import is_torchao_available

is_torchao_available.cache_clear()
print("TorchAO available:", is_torchao_available())
~~~

**Saved output excerpt:**

~~~text
TorchAO available: False

~~~

### Yesterday's notebook — cell 26

~~~python
import json
import time
import torch
from peft import get_peft_model
from transformers import Trainer, TrainingArguments

model_fp16 = get_peft_model(base_model_fp16, lora_config_fp16)
model_fp16.print_trainable_parameters()

fp16_trainable = sum(
    p.numel() for p in model_fp16.parameters() if p.requires_grad
)
print("Trainable parameters:", fp16_trainable)
print("Under 50M limit:", fp16_trainable <= 50_000_000)

FP16_RUN_DIR = DRIVE_ROOT / "runs" / "clean_lora_fp16_r16_e2"
FP16_CHECKPOINT_DIR = FP16_RUN_DIR / "checkpoints"
FP16_ADAPTER_DIR = FP16_RUN_DIR / "adapter"
FP16_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FP16_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

fp16_args = TrainingArguments(
    output_dir=str(FP16_CHECKPOINT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42,
)

trainer_fp16 = Trainer(
    model=model_fp16,
    args=fp16_args,
    train_dataset=train_examples,
    data_collator=collate_batch,
)

torch.cuda.reset_peak_memory_stats()
fp16_start = time.time()

fp16_train_result = trainer_fp16.train()

fp16_minutes = (time.time() - fp16_start) / 60
fp16_peak_allocated = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
fp16_peak_reserved = torch.cuda.max_memory_reserved(0) / (1024 ** 3)

trainer_fp16.model.save_pretrained(
    FP16_ADAPTER_DIR,
    safe_serialization=True,
)

fp16_summary = {
    "model_id": MODEL_ID,
    "data": "cleaned_train.jsonl",
    "method": "16-bit LoRA",
    "rank": 16,
    "epochs": 2,
    "sequence_length": MAX_LENGTH,
    "learning_rate": 2e-4,
    "trainable_parameters": fp16_trainable,
    "training_minutes": fp16_minutes,
    "peak_gpu_allocated_gb": fp16_peak_allocated,
    "peak_gpu_reserved_gb": fp16_peak_reserved,
    "trainer_metrics": fp16_train_result.metrics,
    "training_log": trainer_fp16.state.log_history,
}

with (FP16_RUN_DIR / "run_summary.json").open("w", encoding="utf-8") as f:
    json.dump(fp16_summary, f, indent=2, default=str)

print("Training minutes:", round(fp16_minutes, 1))
print("Peak GPU memory allocated (GB):", round(fp16_peak_allocated, 2))
print("Peak GPU memory reserved (GB):", round(fp16_peak_reserved, 2))
print("Adapter saved to:", FP16_ADAPTER_DIR)
print("Summary saved to:", FP16_RUN_DIR / "run_summary.json")
~~~

**Saved output excerpt:**

~~~text
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Trainable parameters: 18464768
Under 50M limit: True


<IPython.core.display.HTML object>

Training minutes: 25.2
Peak GPU memory allocated (GB): 8.71
Peak GPU memory reserved (GB): 13.64
Adapter saved to: /content/drive/MyDrive/nimbuspay_solution/runs/clean_lora_fp16_r16_e2/adapter
Summary saved to: /content/drive/MyDrive/nimbuspay_solution/runs/clean_lora_fp16_r16_e2/run_summary.json

~~~

### Yesterday's notebook — cell 27

~~~python
import json
import subprocess
import sys
import torch
from tqdm.auto import tqdm

model_fp16.eval()
model_fp16.gradient_checkpointing_disable()
model_fp16.config.use_cache = True

fp16_dev_predictions = []

for row in tqdm(dev_rows, desc="Scoring 16-bit LoRA on dev"):
    messages = row["messages"][:2]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model_fp16.device)

    with torch.inference_mode():
        output_ids = model_fp16.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[1]
    output_text = tokenizer.decode(
        output_ids[0][prompt_length:],
        skip_special_tokens=True,
    )

    fp16_dev_predictions.append({
        "id": row["id"],
        "output": output_text,
    })

fp16_dev_path = DRIVE_ROOT / "clean_lora_fp16_r16_e2_dev.jsonl"

with fp16_dev_path.open("w", encoding="utf-8") as f:
    for item in fp16_dev_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

score_result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "score.py"),
        "--gold", str(ROOT / "data" / "dev.jsonl"),
        "--pred", str(fp16_dev_path),
        "--json",
    ],
    capture_output=True,
    text=True,
    check=True,
)

print(score_result.stdout)
print("Dev predictions saved to:", fp16_dev_path)
~~~

**Saved output excerpt:**

~~~text
Scoring 16-bit LoRA on dev:   0%|          | 0/200 [00:00<?, ?it/s]

{
 "all": {
  "n": 200,
  "json_valid": 1.0,
  "exact_match": 0.805,
  "mean_field_acc": 0.966875,
  "fields": {
   "category": 0.965,
   "priority": 0.895,
   "amount_paise": 0.955,
   "txn_date": 0.985,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.935
  }
 },
 "category=other": {
  "n": 16,
  "json_valid": 1.0,
  "exact_match": 0.6875,
  "mean_field_acc": 0.953125,
  "fields": {
   "category": 0.875,
   "priority": 0.9375,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.8125
  }
 },
 "category=kyc": {
  "n": 24,
  "json_valid": 1.0,
  "exact_match": 1.0,
  "mean_field_acc": 1.0,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=refund": {
  "n": 55,
  "json_valid": 1.0,
  "exact_match": 0.7454545454545455,
  "mean_field_acc": 0.9545454545454546,
  "fields": {
   "category": 1.0,
   "priority": 0.8,
   "amount_paise": 0.9818181818181818,
   "txn_date": 0.9818181818181818,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.8727272727272727
  }
 },
 "category=payment_failed": {
  "n": 53,
  "json_valid": 1.0,
  "exact_match": 0.7735849056603774,
  "mean_field_acc": 0.9669811320754718,
  "fields": {
   "category": 0.9622641509433962,
   "priority": 0.8867924528301887,
   "amount_paise": 0.9245283018867925,
   "txn_date": 0.9811320754716981,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.9811320754716981
  }
 },
 "category=fraud": {
  "n": 15,
  "json_valid": 1.0,
  "exact_match": 0.7333333333333333,
  "mean_field_acc": 0.925,
  "fields": {
   "category": 0.8,
   "priority": 0.8,
   "amount_paise": 0.8666666666666667,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.9333333333333333
  }
 },
 "category=offers": {
  "n": 18,
  "json_valid": 1.0,
  "exact_match": 0.7777777777777778,
  "mean_field_acc": 0.9722222222222222,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 0.8888888888888888,
   "txn_date": 0.9444444444444444,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.9444444444444444
  }
 },
 "category=account_access": {
  "n": 19,
  "json_valid": 1.0,
  "exact_match": 1.0,
  "mean_field_acc": 1.0,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 }
}

Dev predictions saved to: /content/drive/MyDrive/nimbuspay_solution/clean_lora_fp16_r16_e2_dev.jsonl

~~~

### Yesterday's notebook — cell 28

~~~python
import gc
import json
import time
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# Release the previous 16-bit model to free GPU memory.
for name in ["trainer_fp16", "model_fp16", "base_model_fp16"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

base_model_r8 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)
base_model_r8.config.use_cache = False
base_model_r8.gradient_checkpointing_enable()
base_model_r8.enable_input_require_grads()

lora_config_r8 = LoraConfig(
    r=8,                       # changed from rank 16
    lora_alpha=32,             # held constant
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model_r8 = get_peft_model(base_model_r8, lora_config_r8)
model_r8.print_trainable_parameters()

r8_trainable = sum(
    p.numel() for p in model_r8.parameters() if p.requires_grad
)
print("Trainable parameters:", r8_trainable)
print("Under 50M limit:", r8_trainable <= 50_000_000)

R8_RUN_DIR = DRIVE_ROOT / "runs" / "clean_lora_fp16_r8_e2"
R8_CHECKPOINT_DIR = R8_RUN_DIR / "checkpoints"
R8_ADAPTER_DIR = R8_RUN_DIR / "adapter"
R8_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
R8_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

r8_args = TrainingArguments(
    output_dir=str(R8_CHECKPOINT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42,
)

trainer_r8 = Trainer(
    model=model_r8,
    args=r8_args,
    train_dataset=train_examples,
    data_collator=collate_batch,
)

torch.cuda.reset_peak_memory_stats()
r8_start = time.time()

r8_train_result = trainer_r8.train()

r8_minutes = (time.time() - r8_start) / 60
r8_peak_allocated = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
r8_peak_reserved = torch.cuda.max_memory_reserved(0) / (1024 ** 3)

trainer_r8.model.save_pretrained(
    R8_ADAPTER_DIR,
    safe_serialization=True,
)

r8_summary = {
    "model_id": MODEL_ID,
    "data": "cleaned_train.jsonl",
    "method": "16-bit LoRA",
    "rank": 8,
    "epochs": 2,
    "sequence_length": MAX_LENGTH,
    "learning_rate": 2e-4,
    "trainable_parameters": r8_trainable,
    "training_minutes": r8_minutes,
    "peak_gpu_allocated_gb": r8_peak_allocated,
    "peak_gpu_reserved_gb": r8_peak_reserved,
    "trainer_metrics": r8_train_result.metrics,
    "training_log": trainer_r8.state.log_history,
}

with (R8_RUN_DIR / "run_summary.json").open("w", encoding="utf-8") as f:
    json.dump(r8_summary, f, indent=2, default=str)

print("Rank-8 training minutes:", round(r8_minutes, 1))
print("Peak GPU memory allocated (GB):", round(r8_peak_allocated, 2))
print("Peak GPU memory reserved (GB):", round(r8_peak_reserved, 2))
print("Adapter saved to:", R8_ADAPTER_DIR)
print("Summary saved to:", R8_RUN_DIR / "run_summary.json")
~~~

**Saved output excerpt:**

~~~text
Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945
Trainable parameters: 9232384
Under 50M limit: True


<IPython.core.display.HTML object>

Rank-8 training minutes: 24.5
Peak GPU memory allocated (GB): 9.08
Peak GPU memory reserved (GB): 14.0
Adapter saved to: /content/drive/MyDrive/nimbuspay_solution/runs/clean_lora_fp16_r8_e2/adapter
Summary saved to: /content/drive/MyDrive/nimbuspay_solution/runs/clean_lora_fp16_r8_e2/run_summary.json

~~~

### Yesterday's notebook — cell 29

~~~python
import json
import subprocess
import sys
import torch
from tqdm.auto import tqdm

model_r8.eval()
model_r8.gradient_checkpointing_disable()
model_r8.config.use_cache = True

r8_dev_predictions = []

for row in tqdm(dev_rows, desc="Scoring rank-8 LoRA on dev"):
    messages = row["messages"][:2]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model_r8.device)

    with torch.inference_mode():
        output_ids = model_r8.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[1]
    output_text = tokenizer.decode(
        output_ids[0][prompt_length:],
        skip_special_tokens=True,
    )

    r8_dev_predictions.append({
        "id": row["id"],
        "output": output_text,
    })

r8_dev_path = DRIVE_ROOT / "clean_lora_fp16_r8_e2_dev.jsonl"

with r8_dev_path.open("w", encoding="utf-8") as f:
    for item in r8_dev_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

score_result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "score.py"),
        "--gold", str(ROOT / "data" / "dev.jsonl"),
        "--pred", str(r8_dev_path),
        "--json",
    ],
    capture_output=True,
    text=True,
    check=True,
)

print(score_result.stdout)
print("Dev predictions saved to:", r8_dev_path)
~~~

**Saved output excerpt:**

~~~text
Scoring rank-8 LoRA on dev:   0%|          | 0/200 [00:00<?, ?it/s]

{
 "all": {
  "n": 200,
  "json_valid": 1.0,
  "exact_match": 0.83,
  "mean_field_acc": 0.973125,
  "fields": {
   "category": 0.975,
   "priority": 0.92,
   "amount_paise": 0.96,
   "txn_date": 0.99,
   "txn_id": 1.0,
   "channel": 0.995,
   "language": 1.0,
   "needs_human": 0.945
  }
 },
 "category=other": {
  "n": 16,
  "json_valid": 1.0,
  "exact_match": 0.6875,
  "mean_field_acc": 0.953125,
  "fields": {
   "category": 0.875,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 0.9375,
   "language": 1.0,
   "needs_human": 0.8125
  }
 },
 "category=kyc": {
  "n": 24,
  "json_valid": 1.0,
  "exact_match": 1.0,
  "mean_field_acc": 1.0,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=refund": {
  "n": 55,
  "json_valid": 1.0,
  "exact_match": 0.8,
  "mean_field_acc": 0.9659090909090909,
  "fields": {
   "category": 1.0,
   "priority": 0.8545454545454545,
   "amount_paise": 0.9818181818181818,
   "txn_date": 0.9818181818181818,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.9090909090909091
  }
 },
 "category=payment_failed": {
  "n": 53,
  "json_valid": 1.0,
  "exact_match": 0.7735849056603774,
  "mean_field_acc": 0.964622641509434,
  "fields": {
   "category": 0.9622641509433962,
   "priority": 0.8679245283018868,
   "amount_paise": 0.9433962264150944,
   "txn_date": 0.9811320754716981,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.9622641509433962
  }
 },
 "category=fraud": {
  "n": 15,
  "json_valid": 1.0,
  "exact_match": 0.7333333333333333,
  "mean_field_acc": 0.9583333333333334,
  "fields": {
   "category": 0.9333333333333333,
   "priority": 0.9333333333333333,
   "amount_paise": 0.8,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 },
 "category=offers": {
  "n": 18,
  "json_valid": 1.0,
  "exact_match": 0.8888888888888888,
  "mean_field_acc": 0.9861111111111112,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 0.9444444444444444,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 0.9444444444444444
  }
 },
 "category=account_access": {
  "n": 19,
  "json_valid": 1.0,
  "exact_match": 1.0,
  "mean_field_acc": 1.0,
  "fields": {
   "category": 1.0,
   "priority": 1.0,
   "amount_paise": 1.0,
   "txn_date": 1.0,
   "txn_id": 1.0,
   "channel": 1.0,
   "language": 1.0,
   "needs_human": 1.0
  }
 }
}

Dev predictions saved to: /content/drive/MyDrive/nimbuspay_solution/clean_lora_fp16_r8_e2_dev.jsonl

~~~

### Yesterday's notebook — cell 30

~~~python
import json
from collections import Counter

priority_counts = Counter(
    json.loads(row["messages"][-1]["content"])["priority"]
    for row in cleaned_rows
)

print("Training examples by priority:", dict(priority_counts))
~~~

**Saved output excerpt:**

~~~text
Training examples by priority: {'P3': 791, 'P2': 346, 'P1': 281}

~~~

### Yesterday's notebook — cell 31

~~~python
import gc
import json
import time
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# Double P1/P2 examples; leave P3 examples as-is.
priority_weighted_examples = []
weighted_counts = {"P1": 0, "P2": 0, "P3": 0}

for row, example in zip(cleaned_rows, train_examples):
    label = json.loads(row["messages"][-1]["content"])
    priority = label["priority"]
    copies = 2 if priority in {"P1", "P2"} else 1

    priority_weighted_examples.extend([example] * copies)
    weighted_counts[priority] += copies

print("Priority-weighted examples:", len(priority_weighted_examples))
print("Counts after upweighting:", weighted_counts)

# Release the previous rank-8 model before loading a fresh one.
for name in ["trainer_r8", "model_r8", "base_model_r8"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

base_model_own = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)
base_model_own.config.use_cache = False
base_model_own.gradient_checkpointing_enable()
base_model_own.enable_input_require_grads()

lora_config_own = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model_own = get_peft_model(base_model_own, lora_config_own)
model_own.print_trainable_parameters()

own_trainable = sum(
    p.numel() for p in model_own.parameters() if p.requires_grad
)

OWN_RUN_DIR = DRIVE_ROOT / "runs" / "priority_upsample_fp16_r8"
OWN_CHECKPOINT_DIR = OWN_RUN_DIR / "checkpoints"
OWN_ADAPTER_DIR = OWN_RUN_DIR / "adapter"
OWN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OWN_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

own_args = TrainingArguments(
    output_dir=str(OWN_CHECKPOINT_DIR),
    max_steps=178,  # match the rank-8 reference's optimizer-step count
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42,
)

trainer_own = Trainer(
    model=model_own,
    args=own_args,
    train_dataset=priority_weighted_examples,
    data_collator=collate_batch,
)

torch.cuda.reset_peak_memory_stats()
own_start = time.time()

own_train_result = trainer_own.train()

own_minutes = (time.time() - own_start) / 60
own_peak_allocated = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
own_peak_reserved = torch.cuda.max_memory_reserved(0) / (1024 ** 3)

trainer_own.model.save_pretrained(
    OWN_ADAPTER_DIR,
    safe_serialization=True,
)

own_summary = {
    "model_id": MODEL_ID,
    "data": "cleaned_train.jsonl, P1/P2 examples duplicated once",
    "method": "16-bit LoRA",
    "rank": 8,
    "max_steps": 178,
    "sequence_length": MAX_LENGTH,
    "learning_rate": 2e-4,
    "trainable_parameters": own_trainable,
    "training_minutes": own_minutes,
    "peak_gpu_allocated_gb": own_peak_allocated,
    "peak_gpu_reserved_gb": own_peak_reserved,
    "priority_counts_after_upsampling": weighted_counts,
    "trainer_metrics": own_train_result.metrics,
    "training_log": trainer_own.state.log_history,
}

with (OWN_RUN_DIR / "run_summary.json").open("w", encoding="utf-8") as f:
    json.dump(own_summary, f, indent=2, default=str)

print("Training minutes:", round(own_minutes, 1))
print("Peak GPU memory allocated (GB):", round(own_peak_allocated, 2))
print("Peak GPU memory reserved (GB):", round(own_peak_reserved, 2))
print("Adapter saved to:", OWN_ADAPTER_DIR)
print("Summary saved to:", OWN_RUN_DIR / "run_summary.json")
~~~

**Saved output excerpt:**

~~~text
Priority-weighted examples: 2045
Counts after upweighting: {'P1': 562, 'P2': 692, 'P3': 791}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


<IPython.core.display.HTML object>
~~~

### Yesterday's notebook — cell 32

~~~python
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/nimbuspay_solution")
candidate_root = DRIVE_ROOT / "candidate_pack"

if (candidate_root / "data" / "train.jsonl").exists():
    ROOT = candidate_root
    print("Assessment data found in Drive.")
else:
    print("Full assessment pack not found in Drive backup; we may need to upload colab_input.zip again.")

for run_name in [
    "clean_qlora_r16_e2",
    "raw_qlora_r16_e2",
    "clean_lora_fp16_r16_e2",
    "clean_lora_fp16_r8_e2",
    "priority_upsample_fp16_r8",
]:
    run_dir = DRIVE_ROOT / "runs" / run_name
    print(f"\n{run_name}: exists={run_dir.exists()}")
    if run_dir.exists():
        print("  adapter exists:", (run_dir / "adapter").exists())
        print("  checkpoints:", [
            p.name for p in (run_dir / "checkpoints").glob("checkpoint-*")
        ] if (run_dir / "checkpoints").exists() else [])
~~~

**Saved output excerpt:**

~~~text
Mounted at /content/drive
Full assessment pack not found in Drive backup; we may need to upload colab_input.zip again.

clean_qlora_r16_e2: exists=True
  adapter exists: True
  checkpoints: ['checkpoint-89', 'checkpoint-178']

raw_qlora_r16_e2: exists=True
  adapter exists: True
  checkpoints: ['checkpoint-91', 'checkpoint-182']

clean_lora_fp16_r16_e2: exists=True
  adapter exists: True
  checkpoints: ['checkpoint-89', 'checkpoint-178']

clean_lora_fp16_r8_e2: exists=True
  adapter exists: True
  checkpoints: ['checkpoint-89', 'checkpoint-178']

priority_upsample_fp16_r8: exists=True
  adapter exists: True
  checkpoints: []

~~~

### Yesterday's notebook — cell 33

~~~python
from pathlib import Path
from google.colab import files
import json
import shutil
import zipfile

ROOT = Path("/content/candidate_pack")

if not (ROOT / "data" / "dev.jsonl").exists():
    uploaded = files.upload()  # Select colab_input.zip
    zip_name = next(name for name in uploaded if name.endswith(".zip"))

    with zipfile.ZipFile(zip_name) as z:
        z.extractall(ROOT)

print("Dev file found:", (ROOT / "data" / "dev.jsonl").exists())
print("Test file found:", (ROOT / "data" / "test_inputs.jsonl").exists())

# Back up the pack now so a later runtime disconnect is easier to recover from.
shutil.copytree(ROOT, DRIVE_ROOT / "candidate_pack", dirs_exist_ok=True)

own_run_dir = DRIVE_ROOT / "runs" / "priority_upsample_fp16_r8"
summary_path = own_run_dir / "run_summary.json"
adapter_path = own_run_dir / "adapter"

print("\nOwn experiment summary exists:", summary_path.exists())
print("Own adapter files:", [
    p.name for p in adapter_path.iterdir()
] if adapter_path.exists() else [])

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("Own experiment training minutes:", summary.get("training_minutes"))
~~~

**Saved output excerpt:**

~~~text
<IPython.core.display.HTML object>

Saving colab_input.zip to colab_input.zip
Dev file found: True
Test file found: True

Own experiment summary exists: False
Own adapter files: []

~~~

## Today: priority experiment, final adapter, and test predictions

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

my_drive = Path("/content/drive/MyDrive")
print("Drive folders:")
for item in sorted(my_drive.iterdir()):
    print(item.name)

Mounted at /content/drive
Drive folders:
Colab Notebooks
nimbuspay_solution


In [2]:
from pathlib import Path

root = Path("/content/drive/MyDrive/nimbuspay_solution")

print("Project folder found:", root.exists())
print("\nTop-level items:")
for item in sorted(root.iterdir()):
    print(" -", item.name)

print("\nSaved adapter files:")
adapters = list(root.rglob("adapter_config.json"))
if adapters:
    for item in adapters:
        print(" -", item.parent)
else:
    print(" None found")

print("\nCleaned training files:")
cleaned_files = list(root.rglob("cleaned_train.jsonl"))
if cleaned_files:
    for item in cleaned_files:
        print(" -", item)
else:
    print(" None found")

Project folder found: True

Top-level items:
 - nimbuspay_solution

Saved adapter files:
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/clean_lora_fp16_r16_e2/adapter
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/clean_lora_fp16_r16_e2/checkpoints/checkpoint-89
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/clean_lora_fp16_r16_e2/checkpoints/checkpoint-178
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/raw_qlora_r16_e2/adapter
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/raw_qlora_r16_e2/checkpoints/checkpoint-182
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/raw_qlora_r16_e2/checkpoints/checkpoint-91
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/clean_lora_fp16_r8_e2/adapter
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/clean_lora_fp16_r8_e2/checkpoints/checkpoint-89
 - /content/drive/MyDrive/nimbuspay_sol

In [3]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution")

print("Project folder:", PROJECT)
print("\nImportant files found:")
for filename in [
    "cleaned_train.jsonl",
    "dev.jsonl",
    "test_inputs.jsonl",
    "score.py",
    "clean_lora_fp16_r8_e2_dev.jsonl",
]:
    matches = list(PROJECT.rglob(filename))
    print(f"\n{filename}:")
    for path in matches:
        print(" -", path)
    if not matches:
        print(" - not found")

Project folder: /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution

Important files found:

cleaned_train.jsonl:
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/cleaned_train.jsonl

dev.jsonl:
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/candidate_pack/data/dev.jsonl

test_inputs.jsonl:
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/candidate_pack/data/test_inputs.jsonl

score.py:
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/candidate_pack/score.py

clean_lora_fp16_r8_e2_dev.jsonl:
 - /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/clean_lora_fp16_r8_e2_dev.jsonl


In [4]:
from pathlib import Path

runs = PROJECT / "runs"
print("Run folders:")
for run in sorted(runs.iterdir()):
    if run.is_dir():
        print(f"\n{run.name}")
        print("  Files/folders:", [item.name for item in run.iterdir()])
        checkpoints = list(run.rglob("checkpoint-*"))
        print("  Checkpoints:", [item.name for item in checkpoints])

Run folders:

clean_lora_fp16_r16_e2
  Files/folders: ['run_summary.json', 'adapter', 'checkpoints']
  Checkpoints: ['checkpoint-89', 'checkpoint-178']

clean_lora_fp16_r8_e2
  Files/folders: ['run_summary.json', 'checkpoints', 'adapter']
  Checkpoints: ['checkpoint-89', 'checkpoint-178']

clean_qlora_r16_e2
  Files/folders: ['run_summary.json', 'checkpoints', 'adapter']
  Checkpoints: ['checkpoint-178', 'checkpoint-89']

raw_qlora_r16_e2
  Files/folders: ['run_summary.json', 'adapter', 'checkpoints']
  Checkpoints: ['checkpoint-182', 'checkpoint-91']


In [5]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: False


In [1]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution")
TRAIN_PATH = PROJECT / "cleaned_train.jsonl"
DEV_PATH = PROJECT / "candidate_pack/data/dev.jsonl"
TEST_PATH = PROJECT / "candidate_pack/data/test_inputs.jsonl"

print("Project found:", PROJECT.exists())
print("Cleaned training data found:", TRAIN_PATH.exists())
print("Dev data found:", DEV_PATH.exists())
print("Test inputs found:", TEST_PATH.exists())

Mounted at /content/drive
Project found: True
Cleaned training data found: True
Dev data found: True
Test inputs found: True


In [3]:
%pip uninstall -y torchao

import torch
import transformers
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
PyTorch: 2.14.1+cu130
Transformers: 5.22.1
PEFT: 0.21.2
Accelerate: 1.15.0
GPU: Tesla T4


In [4]:
import json
from collections import Counter

with TRAIN_PATH.open(encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

priority_counts = Counter(
    json.loads(row["messages"][-1]["content"])["priority"]
    for row in rows
)

print("Cleaned rows:", len(rows))
print("Priority counts:", dict(priority_counts))
print("First row ID:", rows[0]["id"])
print("Message roles:", [message["role"] for message in rows[0]["messages"]])

Cleaned rows: 1418
Priority counts: {'P3': 791, 'P2': 346, 'P1': 281}
First row ID: tr-00001
Message roles: ['system', 'user', 'assistant']


In [5]:
weighted_rows = []

for row in rows:
    label = json.loads(row["messages"][-1]["content"])
    weighted_rows.append(row)  # keep every example once

    if label["priority"] in {"P1", "P2"}:
        weighted_rows.append(row)  # include P1/P2 a second time

weighted_counts = Counter(
    json.loads(row["messages"][-1]["content"])["priority"]
    for row in weighted_rows
)

print("Priority-weighted examples:", len(weighted_rows))
print("Counts after upweighting:", dict(weighted_counts))

Priority-weighted examples: 2045
Counts after upweighting: {'P3': 791, 'P2': 692, 'P1': 562}


In [6]:
import json

RUN_DIR = PROJECT / "runs/clean_lora_fp16_r8_e2"

summary_path = RUN_DIR / "run_summary.json"
adapter_config_path = RUN_DIR / "adapter/adapter_config.json"

print("Saved run summary:")
print(summary_path.read_text(encoding="utf-8"))

print("\nSaved adapter configuration:")
print(adapter_config_path.read_text(encoding="utf-8"))

Saved run summary:
{
  "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
  "data": "cleaned_train.jsonl",
  "method": "16-bit LoRA",
  "rank": 8,
  "epochs": 2,
  "sequence_length": 2304,
  "learning_rate": 0.0002,
  "trainable_parameters": 9232384,
  "training_minutes": 24.520902252197267,
  "peak_gpu_allocated_gb": 9.081093311309814,
  "peak_gpu_reserved_gb": 14.001953125,
  "trainer_metrics": {
    "train_runtime": 1470.1128,
    "train_samples_per_second": 1.929,
    "train_steps_per_second": 0.121,
    "total_flos": 4510073520101376.0,
    "train_loss": 0.10740394392207767,
    "epoch": 2.0
  },
  "training_log": [
    {
      "loss": 1.3908721923828125,
      "grad_norm": 1.3147547245025635,
      "learning_rate": 0.00019973630216576544,
      "epoch": 0.11283497884344147,
      "step": 10
    },
    {
      "loss": 0.12142478227615357,
      "grad_norm": 0.7447775602340698,
      "learning_rate": 0.0001967856500492068,
      "epoch": 0.22566995768688294,
      "step": 20
    },
    {
  

In [7]:
import inspect
from transformers import TrainingArguments

available = inspect.signature(TrainingArguments.__init__).parameters
for name in [
    "max_steps",
    "num_train_epochs",
    "learning_rate",
    "warmup_ratio",
    "warmup_steps",
    "eval_strategy",
    "save_strategy",
    "fp16",
    "gradient_checkpointing",
]:
    print(f"{name}: {name in available}")

max_steps: True
num_train_epochs: True
learning_rate: True
warmup_ratio: False
warmup_steps: True
eval_strategy: True
save_strategy: True
fp16: True
gradient_checkpointing: True


In [9]:
train_features = []

for row in weighted_rows:
    messages = row["messages"]

    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
    )

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    if not full_text.startswith(prompt_text):
        raise ValueError(f"Rendered prompt is not the start of the full text: {row['id']}")

    encoded = tokenizer(
        full_text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    full_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    boundary = len(prompt_text)
    answer_start = next(
        (i for i, (start, end) in enumerate(offsets) if end > boundary),
        len(full_ids),
    )

    if answer_start == len(full_ids):
        raise ValueError(f"No answer tokens found: {row['id']}")

    if len(full_ids) > MAX_LENGTH:
        raise ValueError(f"{row['id']} has {len(full_ids)} tokens, over the limit")

    labels = [-100] * answer_start + full_ids[answer_start:]

    train_features.append({
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    })

print("Prepared examples:", len(train_features))
print("Longest example:", max(len(x["input_ids"]) for x in train_features))
print("Prompt tokens masked:", answer_start)
print("Answer tokens used for loss:", sum(token != -100 for token in labels))

Prepared examples: 2045
Longest example: 2202
Prompt tokens masked: 87
Answer tokens used for loss: 42


In [10]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)

base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "gate_proj",
        "up_proj",
        "v_proj",
        "down_proj",
        "k_proj",
        "o_proj",
        "q_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", trainable)
print("Under 50M limit:", trainable < 50_000_000)
print("GPU:", torch.cuda.get_device_name(0))

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945
Trainable parameters: 9232384
Under 50M limit: True
GPU: Tesla T4


In [12]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=2,
    max_steps=178,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=0.0002,
    warmup_steps=0,
    lr_scheduler_type="linear",
    weight_decay=0.0,
    fp16=True,
    gradient_checkpointing=True,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="steps",
    save_steps=35,
    save_total_limit=5,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to="none",
    seed=42,
    run_name="priority_upsample_fp16_r8",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_tickets,
)

print("Training examples:", len(train_dataset))
print("Planned training steps:", training_args.max_steps)
print("Effective batch size:", 16)
print("Checkpoint folder:", CHECKPOINT_DIR)

Training examples: 2045
Planned training steps: 178
Effective batch size: 16
Checkpoint folder: /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/priority_upsample_fp16_r8/checkpoints


In [13]:
import json
import time
import torch

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

train_result = trainer.train()

training_minutes = (time.time() - start_time) / 60

adapter_dir = RUN_DIR / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

summary = {
    "model_id": MODEL_ID,
    "data": "cleaned_train.jsonl with P1/P2 duplicated",
    "experiment": "priority upsampling",
    "method": "16-bit LoRA",
    "rank": 8,
    "epochs_setting": 2,
    "max_steps": 178,
    "sequence_length": MAX_LENGTH,
    "learning_rate": 0.0002,
    "training_examples": len(train_dataset),
    "priority_counts": dict(weighted_counts),
    "trainable_parameters": trainable,
    "steps_completed": trainer.state.global_step,
    "training_minutes": training_minutes,
    "peak_gpu_allocated_gb": torch.cuda.max_memory_allocated() / (1024 ** 3),
    "peak_gpu_reserved_gb": torch.cuda.max_memory_reserved() / (1024 ** 3),
    "trainer_metrics": train_result.metrics,
    "training_log": trainer.state.log_history,
}

summary_path = RUN_DIR / "run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Steps completed:", trainer.state.global_step)
print("Training minutes:", round(training_minutes, 1))
print("Adapter saved to:", adapter_dir)
print("Summary saved to:", summary_path)

Step,Training Loss
10,0.759317
20,0.096115
30,0.058009
40,0.055911
50,0.046612
60,0.047383
70,0.038417
80,0.024707
90,0.025847
100,0.021129


Steps completed: 178
Training minutes: 24.9
Adapter saved to: /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/priority_upsample_fp16_r8/adapter
Summary saved to: /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/priority_upsample_fp16_r8/run_summary.json


In [14]:
import json
import torch
from tqdm.auto import tqdm

dev_rows = [
    json.loads(line)
    for line in DEV_PATH.open(encoding="utf-8")
    if line.strip()
]

PRED_PATH = PROJECT / "priority_upsample_fp16_r8_dev.jsonl"

# Keep completed predictions if the cell needs to be resumed.
completed_ids = set()
if PRED_PATH.exists():
    with PRED_PATH.open(encoding="utf-8") as f:
        for line in f:
            try:
                completed_ids.add(json.loads(line)["id"])
            except (json.JSONDecodeError, KeyError):
                pass

model.eval()
device = next(model.parameters()).device

with PRED_PATH.open("a", encoding="utf-8") as f:
    for row in tqdm(dev_rows, desc="Scoring priority experiment on dev"):
        if row["id"] in completed_ids:
            continue

        inputs = tokenizer.apply_chat_template(
            row["messages"][:2],  # system and ticket only
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        prompt_length = inputs["input_ids"].shape[1]
        output_text = tokenizer.decode(
            generated[0, prompt_length:],
            skip_special_tokens=True,
        ).strip()

        f.write(json.dumps(
            {"id": row["id"], "output": output_text},
            ensure_ascii=False,
        ) + "\n")
        f.flush()

print("Dev predictions saved to:", PRED_PATH)
print("Predictions saved:", len(completed_ids) + sum(
    1 for row in dev_rows if row["id"] not in completed_ids
))

Scoring priority experiment on dev:   0%|          | 0/200 [00:00<?, ?it/s]

Dev predictions saved to: /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/priority_upsample_fp16_r8_dev.jsonl
Predictions saved: 200


In [15]:
import subprocess
import sys

SCORE_SCRIPT = PROJECT / "candidate_pack/score.py"

result = subprocess.run(
    [
        sys.executable,
        str(SCORE_SCRIPT),
        "--gold", str(DEV_PATH),
        "--pred", str(PRED_PATH),
    ],
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)

group                            n   valid   exact  fields
all                            200  100.0%   81.5%   96.9%
category=account_access         19  100.0%   89.5%   98.7%
category=fraud                  15  100.0%   66.7%   91.7%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%   88.9%   98.6%
category=other                  16  100.0%   81.2%   97.7%
category=payment_failed         53  100.0%   75.5%   96.2%
category=refund                 55  100.0%   78.2%   96.4%

per-field accuracy (all rows)
  category        98.0%
  priority        91.0%
  amount_paise    94.5%
  txn_date        98.5%
  txn_id         100.0%
  channel        100.0%
  language       100.0%
  needs_human     93.5%



In [16]:
import csv
import json

BEST_PRED_PATH = PROJECT / "clean_lora_fp16_r8_e2_dev.jsonl"
ERRORS_PATH = PROJECT / "best_dev_errors.csv"

gold_rows = [
    json.loads(line)
    for line in DEV_PATH.open(encoding="utf-8")
    if line.strip()
]
predictions = {
    item["id"]: item["output"]
    for item in (
        json.loads(line)
        for line in BEST_PRED_PATH.open(encoding="utf-8")
        if line.strip()
    )
}

fields = [
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human",
]

errors = []
for row in gold_rows:
    expected = json.loads(row["messages"][-1]["content"])
    predicted = json.loads(predictions[row["id"]])
    wrong = [key for key in fields if expected[key] != predicted.get(key)]

    if wrong:
        errors.append({
            "id": row["id"],
            "wrong_fields": ", ".join(wrong),
            "ticket": row["messages"][1]["content"],
            "expected_wrong_values": json.dumps(
                {key: expected[key] for key in wrong}, ensure_ascii=False
            ),
            "predicted_wrong_values": json.dumps(
                {key: predicted.get(key) for key in wrong}, ensure_ascii=False
            ),
        })

with ERRORS_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=errors[0].keys())
    writer.writeheader()
    writer.writerows(errors)

print("Dev tickets with errors:", len(errors))
print("Full error list saved to:", ERRORS_PATH)

for error in errors[:12]:
    print("\nID:", error["id"])
    print("Wrong fields:", error["wrong_fields"])
    print("Expected:", error["expected_wrong_values"])
    print("Predicted:", error["predicted_wrong_values"])
    print("Ticket:", error["ticket"][:500])

Dev tickets with errors: 34
Full error list saved to: /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/best_dev_errors.csv

ID: dv-00001
Wrong fields: needs_human
Expected: {"needs_human": false}
Predicted: {"needs_human": true}
Ticket: Received: 2026-08-28 (Fri)
Source: app-chat

Good morning, What needs to be done to activate and use a card? My earlier ticket number is #54047. Please look into it.

ID: dv-00007
Wrong fields: amount_paise
Expected: {"amount_paise": 87700}
Predicted: {"amount_paise": 877}
Ticket: Received: 2026-08-16 (Sun)
Source: app-chat

I have a payment that I didn't make. ₹877 is the amount. The txn id is NP-0755176283. It was a UPI transaction. Please help.

ID: dv-00010
Wrong fields: category, priority, needs_human
Expected: {"category": "payment_failed", "priority": "P3", "needs_human": false}
Predicted: {"category": "fraud", "priority": "P1", "needs_human": true}
Ticket: Received: 2026-07-09 (Thu)
Source: email

Subject: Problem with NimbusPay

URG

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Free memory used by the priority experiment
for name in ("trainer", "model", "base_model"):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

BEST_ADAPTER_DIR = PROJECT / "adapter"
assert (BEST_ADAPTER_DIR / "adapter_config.json").exists()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, str(BEST_ADAPTER_DIR))
model.config.use_cache = True
model.eval()

print("Final model loaded from:", BEST_ADAPTER_DIR)
print("GPU:", torch.cuda.get_device_name(0))

In [18]:
import json
import torch
from tqdm.auto import tqdm

test_rows = [
    json.loads(line)
    for line in TEST_PATH.open(encoding="utf-8")
    if line.strip()
]

WORKING_PRED_PATH = (
    PROJECT / "runs/clean_lora_fp16_r8_e2/predictions_test_working.jsonl"
)
WORKING_PRED_PATH.parent.mkdir(parents=True, exist_ok=True)

# Resume completed rows if Colab disconnects and you rerun this cell.
existing = {}
if WORKING_PRED_PATH.exists():
    with WORKING_PRED_PATH.open(encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line)
                existing[item["id"]] = item["output"]
            except (json.JSONDecodeError, KeyError):
                pass

model.eval()
device = next(model.parameters()).device

with WORKING_PRED_PATH.open("a", encoding="utf-8") as f:
    for row in tqdm(test_rows, desc="Generating test predictions"):
        if row["id"] in existing:
            continue

        inputs = tokenizer.apply_chat_template(
            row["messages"][:2],
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        prompt_length = inputs["input_ids"].shape[1]
        output_text = tokenizer.decode(
            generated[0, prompt_length:],
            skip_special_tokens=True,
        ).strip()

        item = {"id": row["id"], "output": output_text}
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        f.flush()
        existing[row["id"]] = output_text

print("Test tickets:", len(test_rows))
print("Predictions saved:", len(existing))
print("Working predictions:", WORKING_PRED_PATH)

Generating test predictions:   0%|          | 0/400 [00:00<?, ?it/s]

Test tickets: 400
Predictions saved: 400
Working predictions: /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/runs/clean_lora_fp16_r8_e2/predictions_test_working.jsonl


In [ ]:
import datetime
import json
import re
import shutil

test_ids = [
    json.loads(line)["id"]
    for line in TEST_PATH.open(encoding="utf-8")
    if line.strip()
]

pred_rows = [
    json.loads(line)
    for line in WORKING_PRED_PATH.open(encoding="utf-8")
    if line.strip()
]

required = {
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human",
}
allowed_categories = {
    "payment_failed", "refund", "fraud", "account_access",
    "kyc", "offers", "other",
}
allowed_channels = {"upi", "card", "netbanking", "wallet"}

pred_ids = [row.get("id") for row in pred_rows]
duplicates = len(pred_ids) - len(set(pred_ids))
missing_ids = set(test_ids) - set(pred_ids)
extra_ids = set(pred_ids) - set(test_ids)

invalid = []

for row in pred_rows:
    try:
        obj = json.loads(row["output"])
    except (json.JSONDecodeError, TypeError, KeyError):
        invalid.append((row.get("id"), "output is not valid JSON"))
        continue

    if not isinstance(obj, dict) or set(obj) != required:
        invalid.append((row.get("id"), "wrong JSON keys"))
        continue

    if obj["category"] not in allowed_categories:
        invalid.append((row["id"], "invalid category"))
    if obj["priority"] not in {"P1", "P2", "P3"}:
        invalid.append((row["id"], "invalid priority"))

    amount = obj["amount_paise"]
    if amount is not None and type(amount) is not int:
        invalid.append((row["id"], "amount_paise is not an integer or null"))

    date = obj["txn_date"]
    if date is not None:
        try:
            datetime.datetime.strptime(date, "%Y-%m-%d")
        except (TypeError, ValueError):
            invalid.append((row["id"], "txn_date is not YYYY-MM-DD or null"))

    txn_id = obj["txn_id"]
    if txn_id is not None and not re.fullmatch(r"NP\d{10}", str(txn_id)):
        invalid.append((row["id"], "invalid txn_id"))

    if obj["channel"] is not None and obj["channel"] not in allowed_channels:
        invalid.append((row["id"], "invalid channel"))
    if obj["language"] not in {"en", "hi-en"}:
        invalid.append((row["id"], "invalid language"))
    if type(obj["needs_human"]) is not bool:
        invalid.append((row["id"], "needs_human is not true/false"))

print("Test inputs:", len(test_ids))
print("Prediction rows:", len(pred_rows))
print("Duplicate IDs:", duplicates)
print("Missing IDs:", len(missing_ids))
print("Unexpected IDs:", len(extra_ids))
print("Invalid outputs/fields:", len(invalid))
print("First problems:", invalid[:10])

if (
    len(test_ids) != 400
    or len(pred_rows) != 400
    or duplicates != 0
    or missing_ids
    or extra_ids
    or invalid
):
    print("Schema issues are listed above; raw outputs are retained unchanged.")

FINAL_PRED_PATH = PROJECT / "predictions_test.jsonl"
shutil.copyfile(WORKING_PRED_PATH, FINAL_PRED_PATH)
print("Validated submission file saved to:", FINAL_PRED_PATH)

In [20]:
import json
import shutil

bad_ids = {"ts-00001", "ts-00241", "ts-00367", "ts-00386"}

test_by_id = {row["id"]: row for row in test_rows}
pred_by_id = {
    row["id"]: row
    for row in (
        json.loads(line)
        for line in WORKING_PRED_PATH.open(encoding="utf-8")
        if line.strip()
    )
}

for ticket_id in sorted(bad_ids):
    print("\nID:", ticket_id)
    print("Ticket:", test_by_id[ticket_id]["messages"][1]["content"])
    print("Raw model output:", pred_by_id[ticket_id]["output"])

# Preserve every model output exactly as generated.
FINAL_PRED_PATH = PROJECT / "predictions_test.jsonl"
shutil.copyfile(WORKING_PRED_PATH, FINAL_PRED_PATH)
print("\nRaw predictions copied to:", FINAL_PRED_PATH)


ID: ts-00001
Ticket: Received: 2026-07-06 (Mon)
Source: app-chat

Bhai, Kisi ne OTP maang ke mere paise nikal liye, yeh maine nahi kiya. App version 6.3.7 hai. Yeh 6 din pehle hua. Transaction ID np6090913363 hai. Please help karo.
Raw model output: {"category":"fraud","priority":"P1","amount_paise":null,"txn_date":"2026-07-00","txn_id":"NP6090913363","channel":null,"language":"hi-en","needs_human":true}

ID: ts-00241
Ticket: Received: 2026-07-22 (Wed)
Source: app-chat

Hi, Petrol pump pe payment nahi hua par message aa gaya ki paise kat gaye. This happened last night. Order number 34847 hai. Dhanyavaad.
Raw model output: {"category":"payment_failed","priority":"P3","amount_paise":null,"txn_date":"2026-07-21","txn_id":"NP34847","channel":null,"language":"hi-en","needs_human":false}

ID: ts-00367
Ticket: Received: 2026-07-22 (Wed)
Source: twitter

@NimbusPayCare I received my American Express in Apple Pay, is there a reason why top up is not functioning properly? I have followed up on 

In [21]:
items_to_check = {
    "Test predictions": PROJECT / "predictions_test.jsonl",
    "Cleaning log": PROJECT / "cleaning_log.csv",
    "README": PROJECT / "README.md",
    "Report": PROJECT / "report.md",
    "Best adapter": PROJECT / "runs/clean_lora_fp16_r8_e2/adapter",
}

for name, path in items_to_check.items():
    print(f"\n{name}: {'FOUND' if path.exists() else 'MISSING'}")
    if path.is_dir():
        print("  Contents:", [item.name for item in path.iterdir()])
    elif path.is_file():
        print("  Size:", path.stat().st_size, "bytes")
        if path.suffix == ".csv":
            print("  CSV lines including header:",
                  sum(1 for _ in path.open(encoding="utf-8")))


Test predictions: FOUND
  Size: 83682 bytes

Cleaning log: FOUND
  Size: 20396 bytes
  CSV lines including header: 239

README: MISSING

Report: MISSING

Best adapter: FOUND
  Contents: ['README.md', 'adapter_config.json', 'adapter_model.safetensors']


In [22]:
import json
from pathlib import Path

print("Run summaries:")
for run in sorted((PROJECT / "runs").iterdir()):
    summary_path = run / "run_summary.json"
    if summary_path.exists():
        s = json.loads(summary_path.read_text(encoding="utf-8"))
        print(
            run.name,
            "| minutes:", round(s.get("training_minutes", 0), 1),
            "| peak allocated GB:", round(s.get("peak_gpu_allocated_gb", 0), 2),
            "| peak reserved GB:", round(s.get("peak_gpu_reserved_gb", 0), 2),
        )

notebook_dir = Path("/content/drive/MyDrive/Colab Notebooks")
print("\nNotebooks in Colab Notebooks:")
for notebook in sorted(notebook_dir.glob("*.ipynb")):
    print(" -", notebook.name)

Run summaries:
clean_lora_fp16_r16_e2 | minutes: 25.2 | peak allocated GB: 8.71 | peak reserved GB: 13.64
clean_lora_fp16_r8_e2 | minutes: 24.5 | peak allocated GB: 9.08 | peak reserved GB: 14.0
clean_qlora_r16_e2 | minutes: 34.7 | peak allocated GB: 6.22 | peak reserved GB: 14.22
priority_upsample_fp16_r8 | minutes: 24.9 | peak allocated GB: 6.97 | peak reserved GB: 13.18
raw_qlora_r16_e2 | minutes: 35.0 | peak allocated GB: 7.11 | peak reserved GB: 14.2

Notebooks in Colab Notebooks:
 - NimbusPay Assessment.ipynb
 - Untitled0.ipynb


In [23]:
import json
from pathlib import Path

notebook_dir = Path("/content/drive/MyDrive/Colab Notebooks")

for filename in ["Untitled0.ipynb", "NimbusPay Assessment.ipynb"]:
    path = notebook_dir / filename
    notebook = json.loads(path.read_text(encoding="utf-8"))

    print(f"\n--- {filename} ---")
    print("Total cells:", len(notebook["cells"]))

    for i, cell in enumerate(notebook["cells"]):
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)

        first_line = next(
            (line.strip() for line in source.splitlines() if line.strip()),
            "(empty cell)",
        )

        print(
            f"{i:02d} [{cell['cell_type']}] {first_line[:120]}"
            f" | saved outputs: {len(cell.get('outputs', []))}"
        )


--- Untitled0.ipynb ---
Total cells: 1
00 [code] (empty cell) | saved outputs: 0

--- NimbusPay Assessment.ipynb ---
Total cells: 26
00 [code] from google.colab import drive | saved outputs: 1
01 [code] from pathlib import Path | saved outputs: 1
02 [code] from pathlib import Path | saved outputs: 1
03 [code] from pathlib import Path | saved outputs: 1
04 [code] import torch | saved outputs: 1
05 [code] import torch | saved outputs: 1
06 [code] from google.colab import drive | saved outputs: 1
07 [code] %pip uninstall -y torchao | saved outputs: 1
08 [code] import json | saved outputs: 1
09 [code] weighted_rows = [] | saved outputs: 1
10 [code] import json | saved outputs: 1
11 [code] import inspect | saved outputs: 1
12 [code] train_features = [] | saved outputs: 1
13 [code] import torch | saved outputs: 5
14 [code] training_args = TrainingArguments( | saved outputs: 1
15 [code] import json | saved outputs: 2
16 [code] import json | saved outputs: 2
17 [code] import subprocess | save

In [25]:
from google.colab import files
uploaded = files.upload()

from pathlib import Path
import shutil

old_notebook = Path("/content/Untitled3.ipynb")
print("Uploaded:", old_notebook.exists())

if old_notebook.exists():
    saved_copy = PROJECT / "Untitled3.ipynb"
    shutil.copy2(old_notebook, saved_copy)
    print("Copied into project:", saved_copy)

Saving Untitled3.ipynb to Untitled3.ipynb
Uploaded: True
Copied into project: /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/Untitled3.ipynb


In [26]:
import copy
import json
import uuid
from pathlib import Path

old_path = PROJECT / "Untitled3.ipynb"
today_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/NimbusPay Assessment.ipynb"
)
draft_path = PROJECT / "notebook_combined_draft.ipynb"

old_nb = json.loads(old_path.read_text(encoding="utf-8"))
today_nb = json.loads(today_path.read_text(encoding="utf-8"))

def section(title):
    return {
        "cell_type": "markdown",
        "metadata": {},
        "source": [f"# {title}\n"],
    }

cells = (
    [section("NimbusPay ticket triage — combined notebook")]
    + old_nb["cells"]
    + [section("Today's continuation: priority experiment and test predictions")]
    + today_nb["cells"]
)

# Give every cell a unique notebook ID.
for cell in cells:
    cell["id"] = uuid.uuid4().hex[:8]

combined = copy.deepcopy(today_nb)
combined["cells"] = cells

draft_path.write_text(
    json.dumps(combined, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Combined draft saved:", draft_path)
print("Total cells:", len(cells))

Combined draft saved: /content/drive/MyDrive/nimbuspay_solution/nimbuspay_solution/notebook_combined_draft.ipynb
Total cells: 65
